# 8: Unsupervised Learning and Clustering

**Author:** Dr Rob Lyon 

**Version:** 1.0

## Code & License
Copyright © 2026 Robert Lyon. All Rights Reserved.

This notebook and all associated materials are the intellectual property of the author.

Permission is granted solely to read, study, and analyse this material for personal educational purposes. No other rights are granted.

Without the prior written consent of the author, you may not:

* Copy, reproduce, redistribute, publish, transmit, or display this work in whole or in part.
* Modify, adapt, transform, translate, or create derivative works based on this material.
* Incorporate any portion of this work into another project, publication, product, model, dataset, or codebase.
* Use this material for commercial purposes.
* Remove or alter this copyright notice.

All intellectual property rights, including copyright and any derivative rights, remain exclusively vested in the author.

Access to this material does not constitute a transfer of ownership, license, or any other intellectual property rights except as expressly stated above.

---

**Note:**

This notebook is designed to be viewed from within JupyterLab. The Table of Contents and "Back to Table of Contents" links rely on JupyterLab's anchor handling to jump between sections.

If you open this notebook in another tool, such as PyCharm or Visual Studio Code, these anchor links may not work as expected. You can still navigate the notebook in those tools by using the headings directly, most editors provide an outline or contents panel that lists them for you.

---

## Table of Contents

1. [Introduction](#1-introduction)
2. [Useful Links](#2-useful-links)
3. [Libraries and Environment Setup](#3-libraries-and-environment-setup)
4. [Unsupervised Learning: Foundations](#4-unsupervised-learning-foundations)
    - [4.1 The Cluster Assumption](#41-the-cluster-assumption)
    - [4.2 The Smoothness Assumption](#42-the-smoothness-assumption)
    - [4.3 The Manifold Assumption](#43-the-manifold-assumption)
    - [4.4 From Unsupervised to Supervised: Pseudo-labelling](#44-from-unsupervised-to-supervised-pseudo-labelling)
5. [Similarity and Distance Metrics](#5-similarity-and-distance-metrics)
    - [5.1 Euclidean Distance](#51-euclidean-distance)
    - [5.2 Manhattan Distance](#52-manhattan-distance)
    - [5.3 Minkowski Distance](#53-minkowski-distance)
    - [5.4 Mahalanobis Distance](#54-mahalanobis-distance)
    - [5.5 Jaccard Similarity](#55-jaccard-similarity)
    - [5.6 Cosine Similarity](#56-cosine-similarity)
    - [5.7 How Metric Choice Affects Nearest Neighbours](#57-how-metric-choice-affects-nearest-neighbours)
6. [k-Nearest Neighbours (k-NN)](#6-k-nearest-neighbours-k-nn)
    - [6.1 From Scratch — Pure Python](#61-from-scratch--pure-python)
    - [6.2 k-NN with scikit-learn](#62-k-nn-with-scikit-learn)
7. [K-Means Clustering](#7-k-means-clustering)
    - [7.1 From Scratch — Pure Python](#71-from-scratch--pure-python)
    - [7.2 Choosing k: The Elbow Method](#72-choosing-k-the-elbow-method)
    - [7.3 K-Means with scikit-learn](#73-k-means-with-scikit-learn)
8. [Hierarchical Clustering](#8-hierarchical-clustering)
    - [8.1 Linkage Criteria](#81-linkage-criteria)
    - [8.2 Dendrogram — then what?](#82-dendrogram--then-what)
    - [8.3 Hierarchical Clustering with scikit-learn](#83-hierarchical-clustering-with-scikit-learn)
9. [DBSCAN](#9-dbscan)
    - [9.1 DBSCAN with scikit-learn](#91-dbscan-with-scikit-learn)
10. [Summary](#10-summary)
11. [References](#11-references)



---

## 1. Introduction

Welcome to **Notebook 8** which introduces **unsupervised learning** — the branch of machine learning that discovers hidden structure in data without relying on labelled examples. The aim is to build a thorough understanding of unsupervised learning, from its theoretical foundations and the distance metrics that underpin common algorithms, through to hands-on implementation of four of the most important clustering methods.

### Learning Objectives

By the end of this notebook you should be able to:

- Describe the three assumptions (cluster, smoothness, manifold) that underpin unsupervised learning and explain how they motivate different algorithms.
- Explain the concept of pseudo-labelling and how it bridges unsupervised and supervised learning.
- Compute and compare Euclidean, Manhattan, Minkowski, Mahalanobis, Jaccard, and Cosine distance/similarity measures, and justify which metric is appropriate for a given problem.
- Implement k-Nearest Neighbours (k-NN) and K-Means clustering from scratch in pure Python.
- Use scikit-learn to fit k-NN, K-Means, Hierarchical Clustering, and DBSCAN models on real and synthetic data.
- Apply the elbow method to select the number of clusters for K-Means.
- Interpret a dendrogram and compare the effect of different linkage criteria on cluster shapes.
- Tune DBSCAN's $\varepsilon$ and `minPts` parameters and explain how density-based clustering differs from centroid-based clustering.




---


## 2. Useful Links

| Link | Description |
|------|-------------|
| [scikit-learn: clustering](https://scikit-learn.org/stable/modules/clustering.html) | Official scikit-learn documentation covering all clustering algorithms used in this notebook, including K-Means, Hierarchical Clustering, and DBSCAN — with worked examples and a comparison guide. |
| [scikit-learn: neighbours](https://scikit-learn.org/stable/modules/neighbors.html) | Documentation for `KNeighborsClassifier` and related classes. Covers distance metrics, algorithm backends (ball-tree, kd-tree), and usage examples. |
| [SciPy: spatial distance](https://docs.scipy.org/doc/scipy/reference/spatial.distance.html) | Reference for the `cdist` and `pdist` functions used to compute pairwise distances, with all supported metrics documented. |
| [SciPy: hierarchical clustering](https://docs.scipy.org/doc/scipy/reference/cluster.hierarchy.html) | Documentation for `linkage` and `dendrogram` — the functions used to build and visualise agglomerative clustering trees. |
| [NumPy documentation](https://numpy.org/doc/stable/) | Reference for the array operations used throughout this notebook, including `np.linalg`, `np.random`, and broadcasting rules. |
| [Matplotlib documentation](https://matplotlib.org/stable/) | Reference for all plotting functions. Particularly useful for understanding `plt.subplots`, `ax.scatter`, and `ax.contourf`. |

[Back to Table of Contents](#Table-of-Contents)



---

## 3. Libraries and Environment Setup

🔙 [Back to Table of Contents](#Table-of-Contents)


### 3.1 Working Environment

To run the code in this notebook, you will need a Python environment with the required libraries listed below installed.

The easiest way to get started is to use the **Project IO** Docker container, which provides a fully configured and tested environment containing all required Python packages and supporting dependencies. Instructions for downloading and running the container can be found in the **project-io** GitHub repository:

https://github.com/scienceguyrob/project-io

Using the Docker container ensures that the notebooks run in exactly the same environment in which they were developed and tested.

If you prefer not to use Docker, a good alternative is to install the [Anaconda Distribution](https://www.anaconda.com/products/distribution). You can then create a new Python environment and install the required libraries using either `conda install` or `pip install`.

### 3.2 Libraries Used in This Notebook

All sections of this notebook use external libraries. The table below lists each library and explains why it is needed.

| Library | Purpose | Documentation |
|---------|---------|---------------|
| **NumPy** (`numpy`) | Fast numerical arrays, linear algebra, and random number generation. Used throughout for distance calculations and data generation. | [numpy.org](https://numpy.org/doc/stable/) |
| **Matplotlib** (`matplotlib.pyplot`) | The standard Python plotting library. Used for all scatter plots, line plots, and decision-boundary visualisations. | [matplotlib.org](https://matplotlib.org/stable/) |
| **Seaborn** (`seaborn`) | High-level statistical visualisation built on Matplotlib. Used to set the visual theme for all plots. | [seaborn.pydata.org](https://seaborn.pydata.org/) |
| **SciPy** (`scipy`) | Scientific computing routines. We use `cdist` for pairwise distances and `linkage`/`dendrogram` for hierarchical clustering. | [scipy.org](https://docs.scipy.org/doc/scipy/) |
| **scikit-learn** (`sklearn`) | The primary machine learning library. Provides `KNeighborsClassifier`, `KMeans`, `AgglomerativeClustering`, `DBSCAN`, and dataset generators. | [scikit-learn.org](https://scikit-learn.org/stable/) |
| **collections** | Python standard library. We use `Counter` for majority-vote label propagation in the pseudo-labelling example. | [docs.python.org](https://docs.python.org/3/library/collections.html) |
| **ipywidgets** (`ipywidgets`) | Adds interactive UI controls (sliders, dropdowns, etc.) to Jupyter notebooks. We use it to create sliders for adjusting plot parameters in real time. | [ipywidgets.readthedocs.io](https://ipywidgets.readthedocs.io/en/stable/) |
| **ipympl** (`matplotlib widget` backend) | Enables interactive, live-updating Matplotlib figures inside Jupyter. Required for smooth slider updates without redrawing the entire plot. | [matplotlib.org/ipympl](https://matplotlib.org/ipympl/) |

### 3.3 Imports

All library imports for this notebook are placed in the cell below. This is a deliberate best practice — keeping all imports in one place at the top of a notebook makes it easy to see at a glance what is required and avoids confusing errors that arise when an import cell is skipped.

> ⚠️ **You must run the cell below before executing any other code cell in this notebook.** If you skip this cell, all subsequent cells will raise a `NameError`.

In [ ]:
# Listing 1.
# ─────────────────────────────────────────────────────────────────────────────
# All library imports for this notebook are placed here for clarity.
# Run this cell first before executing any code.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np                          # Numerical arrays and linear algebra
import matplotlib.pyplot as plt             # Plotting and visualisation
import matplotlib.cm as cm                  # Colour maps for multi-cluster plots
import matplotlib.patches as mpatches       # Patch shapes used in legend / annotations

import matplotlib
matplotlib.rcParams['figure.max_open_warning'] = 50

import seaborn as sns                       # Plotting theme and global style
from collections import Counter             # Majority-vote counting for pseudo-labelling

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster  # Hierarchical clustering

from sklearn.datasets import make_blobs, make_moons, make_circles  # Synthetic datasets
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# A single seeded random number generator used throughout for reproducibility
rng = np.random.default_rng(42)

# Confirm successful import
print('Libraries loaded successfully.')



---

## 4. Unsupervised Learning: Foundations

🔙 [Back to Table of Contents](#table-of-contents)

In supervised learning — covered in Notebooks 5 and 6 — every training example carried a label: a target value telling the algorithm what the correct answer was. The model's job was to learn a mapping from inputs to those labels. In **unsupervised learning** we remove that scaffold entirely. We have only the raw data matrix $\mathbf{X}$ — no target column, no ground truth, no teacher. The algorithm must find structure on its own.

This might sound like a harder problem, and in some ways it is. But it is also a more honest description of most real-world data: labels are expensive, slow to produce, and require human expertise. Raw data, by contrast, is abundant. Unsupervised methods let us ask: *what is this data actually doing?* — before we ever decide what we want to predict.

The kinds of structure we might discover include: natural groupings of similar observations (clustering), compact representations that discard noise while preserving the signal (dimensionality reduction), and probability models that describe how the data was generated (density estimation). This notebook focuses on clustering; Notebook 9 takes up dimensionality reduction.

> 📓 **Notebook 5 & 6 recap:** In Notebook 5 we fitted linear and logistic regression models to labelled data. In Notebook 6 we saw how Naïve Bayes, decision trees, and SVMs each define a different kind of supervised decision boundary. All of those methods require $(X, y)$ pairs at training time. Everything in this notebook works from $\mathbf{X}$ alone.

Before we study any individual algorithm, it is worth understanding the three core assumptions that underpin nearly all unsupervised methods. These assumptions are not always stated explicitly, but they are always present — and knowing them tells you immediately when a method is likely to succeed and when it is likely to fail.

---

### 4.1 The Cluster Assumption

Data naturally organises itself into compact, well-separated groups. Points within a cluster are more similar to each other than to points outside it. If this assumption holds, a good clustering algorithm should be able to recover those groups by measuring similarity or distance between points.

Methods that rely on this assumption — K-Means, Hierarchical Clustering, DBSCAN — will each be covered in detail later in this notebook. They differ in *how* they define a cluster (spherical blobs, a tree of merges, or dense regions separated by low-density gaps), but all of them require the data to have some meaningful grouping structure in the first place. If the data is uniformly distributed with no natural groups, every clustering algorithm will still return clusters — they just will not mean anything.

Clustering algorithms always produce output, even on data with no real cluster structure. The elbow method and silhouette score are tools for detecting when a clustering solution is genuine rather than arbitrary — always validate your results rather than trusting the partition at face value.

---

### 4.2 The Smoothness Assumption

A small change in the input produces a small change in the output. In geometric terms: nearby points in feature space are likely to belong to the same class or have similar target values. This assumption does not require discrete clusters — it simply requires that the data surface is not wildly erratic.

This is the assumption that justifies k-Nearest Neighbours (k-NN), covered later in this notebook. If the smoothness assumption holds, then a point's neighbours are genuinely informative about its identity. If it does not — for example, in data where boundaries are highly irregular or noisy — k-NN degrades quickly.

The error landscapes and gradient descent visualisations in Notebook 4 are a concrete example of the smoothness assumption at work. A smooth, bowl-shaped loss surface is precisely what makes gradient-based optimisation tractable.

---

### 4.3 The Manifold Assumption

High-dimensional data rarely uses all the dimensions available to it. In practice, the meaningful variation in a dataset tends to live on (or very near) a much lower-dimensional curved surface — a **manifold** — embedded within the full high-dimensional space.

A concrete example: a photograph of a face might contain $256 \times 256 = 65{,}536$ pixel values, giving a nominal dimensionality of 65,536. But the axes of *meaningful* variation — pose, expression, lighting angle, identity — number perhaps in the dozens. The manifold assumption says that all the variation we actually care about can be described by a small number of underlying factors, and the high-dimensional observations are just a noisy projection of that lower-dimensional reality.

This has a practical consequence: if the manifold assumption holds, we can compress our data dramatically without losing the structure that matters. Methods that exploit this — PCA, t-SNE, UMAP — are the subject of Notebook 9. For now it is enough to hold the intuition: more dimensions does not mean more information, and the curse of dimensionality (introduced in Notebook 4) is precisely the cost of ignoring this.

---

The cell below visualises all three assumptions simultaneously, so that their distinctions are concrete before we study any individual algorithm. The left panel shows tight, separated blobs — the cluster assumption. The centre panel shows a smoothly varying colour gradient across the space — the smoothness assumption. The right panel shows a low-dimensional curve embedded in three dimensions — the manifold assumption. None of these panels has labels. The structure is entirely in the geometry of the data itself.


In [ ]:
# Listing 2.
%matplotlib widget
from visualisations.Figure_1 import show
show()

### 4.4 From Unsupervised to Supervised: Pseudo-labelling

One of the most practically useful ideas in this notebook sits at the boundary between supervised and unsupervised learning. In Notebooks 5 and 6 we assumed that every training point carried a label. In practice that assumption is often expensive to satisfy — a radiologist annotating medical scans, a linguist tagging rare dialects, or a structural engineer classifying sensor faults all represent bottlenecks where expert time is the limiting factor. Unlabelled data, by contrast, is usually abundant.

**Pseudo-labelling** exploits this asymmetry. The core idea is simple: if the cluster assumption holds and the data naturally organises into groups, then a cluster is a reasonable proxy for a class. We cluster the full dataset — ignoring labels entirely — and then ask: among the handful of points in each cluster whose labels we *do* know, which label is most common? That majority label is then propagated to every other member of the cluster. A dataset that was 95% unlabelled is now fully annotated, at the cost of one clustering pass.

This is not a free lunch. The technique assumes that cluster boundaries and class boundaries align — an assumption that can fail when classes are not well-separated or when the feature space does not reflect the labelling criterion. But where the cluster assumption holds, the results can be striking.

Figure 2 below walks through the process in three steps on a 300-point dataset where only 15 labels (~5%) are revealed. Watch how confidently the majority-label rule recovers the full class structure from almost no supervision. K-Means is used here to define the clusters, is covered later in this notebook. For now it is enough to know that it partitions the data into $k$ groups by minimising the distance of each point to its nearest cluster centre.

In [ ]:
# Listing 3.
%matplotlib widget
from visualisations.Figure_2 import show
show()

---

## 5. Similarity and Distance Metrics

🔙 [Back to Table of Contents](#Table-of-Contents)


Every clustering and neighbourhood algorithm rests on a **distance metric** — a function $d(\mathbf{a}, \mathbf{b})$ that quantifies how similar or dissimilar two points are. The choice of metric is not incidental; it fundamentally shapes what the algorithm considers to be a cluster.

A valid distance metric must satisfy four formal properties:

1. $d(\mathbf{a}, \mathbf{b}) \geq 0$  — non-negativity
2. $d(\mathbf{a}, \mathbf{b}) = 0 \iff \mathbf{a} = \mathbf{b}$  — identity of indiscernibles
3. $d(\mathbf{a}, \mathbf{b}) = d(\mathbf{b}, \mathbf{a})$  — symmetry
4. $d(\mathbf{a}, \mathbf{c}) \leq d(\mathbf{a}, \mathbf{b}) + d(\mathbf{b}, \mathbf{c})$  — triangle inequality

This section introduces six metrics in increasing order of sophistication.

---

### 5.1 Euclidean Distance

The most familiar distance measure is the straight-line distance between two points — what you would measure with a ruler on a piece of paper. For two points $\mathbf{p} = (x_1, y_1)$ and $\mathbf{q} = (x_2, y_2)$ in two dimensions, this is just Pythagoras theorem:

$$d_{\text{Euclidean}}(\mathbf{p}, \mathbf{q}) = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$

In $n$ dimensions the same idea extends naturally — sum the squared difference on each axis and take the square root:

$$d_{\text{Euclidean}}(\mathbf{p}, \mathbf{q}) = \sqrt{\sum_{i=1}^{n}(q_i - p_i)^2}$$

This is the default distance measure in most ML libraries and works well when features are on comparable scales and the data is continuous. It treats all directions in feature space equally: a difference of 1 unit along any axis contributes the same amount to the total distance regardless of which axis it is.

That equal-weighting property is also its main vulnerability. If one feature has a much larger numerical range than another — say, age in years alongside income in pounds — Euclidean distance will be dominated by the high-range feature, effectively ignoring the others. Feature scaling (covered in Notebook 3) is therefore a prerequisite for any distance-based algorithm.

> ⚠️ **Warning:** Euclidean distance is sensitive to the scale of each feature. Always standardise or normalise your features before applying any distance-based algorithm. An unscaled income feature measured in tens of thousands will drown out an age feature measured in tens, even if age is the more informative variable.

Figure 3 below lets you drag two points around a 2D plane. The annotation panel on the right shows the full calculation live, substituting the current coordinates at each step, so you can see exactly how the formula produces the displayed distance.

In [ ]:
# Listing 4.
%matplotlib widget
from visualisations.Figure_3 import show
show()

### 5.2 Manhattan Distance

Euclidean distance travels in a straight line between two points — it cuts diagonally across space regardless of the axes. Manhattan distance takes a different approach: it only moves parallel to the axes, adding up the absolute difference on each dimension separately.

$$d_{\text{Manhattan}}(\mathbf{p}, \mathbf{q}) = \sum_{i=1}^{n} |q_i - p_i|$$

The name comes from the grid layout of Manhattan's street network. If you want to travel from one intersection to another you cannot cut diagonally through a city block — you must walk along the streets, accumulating distance one block at a time. In two dimensions this means one horizontal step and one vertical step; in $n$ dimensions it means one step per axis.

Compared to Euclidean distance, Manhattan distance is less sensitive to large differences on a single axis. Because differences are summed rather than squared, an outlying value on one feature does not dominate the total the way it would under Euclidean distance. This makes Manhattan distance a reasonable choice when your data contains features with occasional large deviations, or when the axes represent genuinely independent, non-interchangeable quantities.

It is less appropriate when the diagonal direction between two points is the meaningful one — for example, in image processing or spatial data where the straight-line path between pixels or coordinates carries a real physical interpretation.

The absolute value operation $|q_i - p_i|$ is the same building block we used when computing mean absolute error in Notebook 3. Both penalise deviations linearly rather than quadratically, which is precisely what makes them robust to large individual differences.

Figure 4 below uses the same draggable interface as Figure 3. As you move either point, the horizontal and vertical steps are drawn separately in different colours, and the annotation panel on the right shows each term in the sum so you can see how the total accumulates across the two axes.

In [ ]:
# Listing 5.
%matplotlib widget
from visualisations.Figure_4 import show
show()

| Metric | Formula | When to use |
|--------|---------|-------------|
| **Euclidean** | $\sqrt{\sum_i (a_i - b_i)^2}$ | Default choice; assumes isotropic space; sensitive to scale |
| **Manhattan** | $\sum_i \|a_i - b_i\|$ | Grid-like movement; less sensitive to outliers than Euclidean |

---

### 5.3 Minkowski Distance

Euclidean and Manhattan distance are not two unrelated ideas — they are both members of a single, unified family. The **Minkowski distance** of order $p$ is defined as:

$$d_p(\mathbf{a}, \mathbf{b}) = \left( \sum_{i=1}^{n} |a_i - b_i|^p \right)^{1/p}$$

Setting $p = 1$ recovers Manhattan distance exactly — differences are summed without any squaring. Setting $p = 2$ recovers Euclidean distance — differences are squared before summing, then square-rooted. The parameter $p$ is therefore a continuous dial that interpolates between these two familiar extremes and beyond.

As $p$ increases, the metric places progressively more weight on the largest coordinate difference and progressively less on the smaller ones. In the limit as $p \to \infty$, all dimensions except the one with the largest difference are effectively ignored, and the distance converges to the **Chebyshev distance**:

$$d_{\infty}(\mathbf{a}, \mathbf{b}) = \max_i |a_i - b_i|$$

Chebyshev distance appears naturally in board games — the number of moves a king needs to travel between two squares on a chessboard is exactly the Chebyshev distance between them, because the king can move diagonally and therefore covers both axes simultaneously in a single move.

The table below summarises the three most practically important cases:

| $p$ value | Name | Formula | Geometric shape of unit ball |
|-----------|------|---------|------------------------------|
| $p = 1$ | Manhattan | $\sum_i \|a_i - b_i\|$ | Diamond (rotated square) |
| $p = 2$ | Euclidean | $\sqrt{\sum_i (a_i - b_i)^2}$ | Circle |
| $p \to \infty$ | Chebyshev | $\max_i \|a_i - b_i\|$ | Square |

The **unit ball** — the set of all points at distance exactly 1 from the origin — is the most revealing way to see what a distance metric is actually doing geometrically. Under Manhattan distance ($p=1$) the unit ball is a diamond: you can spend your distance budget on one axis or spread it across both, but the total absolute displacement is fixed at 1. Under Euclidean distance ($p=2$) it is a circle: all directions are treated equally, and the straight-line reach is constant in every direction. Under Chebyshev ($p \to \infty$) it is a square: what matters is how far you can reach on your worst axis, and you can move freely on all others up to that limit.

Values of $p$ between 1 and 2 produce shapes that transition smoothly from diamond to circle; values above 2 continue toward the square. In practice, $p = 1$ and $p = 2$ cover the vast majority of use cases. Non-integer values of $p$ are occasionally useful in specialised domains but add interpretive complexity without a clear geometric justification in most settings.

> ⚠️ **Warning:** For $p < 1$ the Minkowski formula no longer satisfies the triangle inequality — a basic requirement for a valid distance metric. It can still be used as a similarity measure in some contexts, but you lose the geometric guarantees that distance-based algorithms rely on. Stick to $p \geq 1$ unless you have a specific reason to do otherwise.

The cell below visualises unit balls for several values of $p$, from 0.5 through to $p = \infty$. As $p$ increases, watch the diamond corners fill in toward a circle, and the circle corners fill out toward a square. This shape is the geometric fingerprint of the metric — it tells you exactly which directions in feature space the algorithm treats as equidistant from the origin.

In [ ]:
# Listing 6.
%matplotlib widget
from visualisations.Figure_5 import show
show()

---

### 5.4 Mahalanobis Distance

Every distance metric we have seen so far treats the axes of feature space as independent and equally important. Euclidean distance weights all directions equally; Manhattan distance sums absolute differences without any adjustment for scale. Neither has any awareness of the structure of the data itself.

In real datasets, features are rarely independent. Height and weight are positively correlated. Income and years of experience move together. When two features are strongly correlated, most of the data lives along a diagonal axis — and a step in the perpendicular direction, away from that axis, is far more unusual than a step of the same size along it. Euclidean distance cannot see this distinction. **Mahalanobis distance** can.

The idea is to measure distance not in the raw feature space, but in a transformed space where the data has been rescaled and decorrelated. This transformation is encoded in the **covariance matrix** $\mathbf{S}$, which we first encountered in Notebook 3. The Mahalanobis distance from a point $\mathbf{x}$ to a reference point $\boldsymbol{\mu}$ (typically the mean of the distribution) is:

$$d_{\text{M}}(\mathbf{x}, \boldsymbol{\mu}) = \sqrt{(\mathbf{x} - \boldsymbol{\mu})^\top \mathbf{S}^{-1} (\mathbf{x} - \boldsymbol{\mu})}$$

The term $\mathbf{S}^{-1}$ is the inverse of the covariance matrix. Pre-multiplying and post-multiplying by it has two simultaneous effects. First, it **normalises each feature** by its variance, so a feature with a large range does not dominate the distance — the same scale-invariance we would achieve by standardising manually. Second, it **accounts for correlations**: a displacement in a direction of high variance (along the principal axis of the data) contributes less to the distance than a displacement of the same magnitude in a direction of low variance (perpendicular to it). Moving away from the data cloud in an unusual direction is penalised more heavily than moving the same distance within it.

This has a useful geometric interpretation. The **unit ball** under Mahalanobis distance is an ellipse (in 2D) or an ellipsoid (in higher dimensions) whose axes are aligned with the principal directions of the data and whose radii are proportional to the standard deviations along those directions. Two points that lie on the same ellipse are considered equally distant from the mean — even if their Euclidean distances are different. A point that lies far from the data's principal axis, in a direction the data rarely occupies, will have a large Mahalanobis distance even if it is Euclidean-close to the mean.

This makes Mahalanobis distance particularly well-suited to **anomaly detection**. Think of it this way: if you work in a hospital and you know that height and weight are strongly correlated in your patient population, then a patient who is very tall but very light is unusual in a way that raw Euclidean distance might miss — because Euclidean distance has no knowledge of that correlation. The Mahalanobis distance does. A large Mahalanobis distance from the centre of the distribution is a signal that a point is genuinely surprising given the shape of the data, not just far away in a raw numerical sense. The flip side is that Mahalanobis distance needs a well-defined reference distribution to measure surprise against. If there is no natural centre and covariance structure — for example, when comparing arbitrary pairs of points drawn from different populations — the metric loses its meaning.

> ⚠️ **Warning:** Mahalanobis distance requires inverting the covariance matrix $\mathbf{S}$. If any features are perfectly correlated, or if you have more features than data points, $\mathbf{S}$ will be singular and cannot be inverted. In practice this means Mahalanobis distance is unreliable in very high-dimensional settings unless you first reduce dimensionality.

Figure 6 below makes this concrete. We generate 300 points from a **bivariate normal distribution** — a generalisation of the familiar bell curve to two dimensions. Where a standard normal distribution describes the spread of a single variable, a bivariate normal describes the joint spread of two variables simultaneously, including how they relate to each other. Here the two features are positively correlated, so the data cloud runs diagonally: knowing that feature 1 is large tells you that feature 2 is likely large too. This diagonal structure is exactly what the covariance matrix $\mathbf{S}$ encodes, and what Euclidean distance is blind to.

A single red test point is draggable anywhere on the plot. As you move it, the right-hand panel updates with the full calculation: the difference vector $\mathbf{x} - \boldsymbol{\mu}$, the estimated inverse covariance matrix $\mathbf{S}^{-1}$, the intermediate products, and the final Mahalanobis distance — alongside the Euclidean distance for direct comparison.

Try dragging the point to two positions that are the same Euclidean distance from the centre but in different directions. Place it first along the main diagonal axis of the data cloud, then move it to a position perpendicular to that axis at the same Euclidean distance. The Euclidean distance will be identical in both cases. The Mahalanobis distance will not — the perpendicular position sits across the grain of the data, in a region the distribution rarely reaches, and is penalised accordingly. The annotation panel shows exactly which step in the calculation drives the difference: it is the multiplication by $\mathbf{S}^{-1}$ that stretches the space, compressing directions of high variance and expanding directions of low variance, so that distance reflects statistical unusualness rather than raw geometry.


In [ ]:
# Listing 7.
%matplotlib widget
from visualisations.Figure_6 import show
show()

### 5.5 Jaccard Similarity

All the metrics we have covered so far operate on continuous numerical features. But many real datasets are binary — each feature records simply whether something is present or absent. A document either contains a word or it does not. A patient either has a symptom or they do not. A user either purchased a product or they did not.

Applying Euclidean or Manhattan distance to binary data produces a subtle problem. When two data points both lack a feature — two documents that neither contains the word "elephant", for example — that shared absence looks like evidence of similarity. In most cases it is not: there are thousands of words neither document contains, and counting all those shared absences inflates the similarity between things that may have nothing meaningful in common.

**Jaccard similarity** sidesteps this entirely by only looking at positive matches. Given two sets $A$ and $B$, it is defined as the size of their intersection divided by the size of their union:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

In plain terms: of all the features that are present in at least one of the two points, what fraction are present in both? Shared absences never appear in either the numerator or the denominator, so they contribute nothing to the score. The result ranges from 0 (no features in common) to 1 (identical feature sets).

Jaccard similarity is widely used in text mining (comparing documents by the words they contain), genomics (comparing gene expression profiles), and recommendation systems (comparing users by the items they have interacted with). It is less appropriate for continuous or count data, where the magnitude of each feature carries meaningful information that a binary present/absent encoding would discard.

The Jaccard similarity is actually a *similarity* measure, not a distance. To use it with distance-based algorithms like k-NN or hierarchical clustering, we must convert it to a distance via $d_J = 1 - J(A, B)$. This is known as the Jaccard distance and satisfies the triangle inequality, making it a valid metric.

---

Figure 7 below lets you explore how the Jaccard calculation responds to changes in set composition. There are three sliders. The first two control how many of the 20 features are active (set to 1) in vector A and vector B respectively. The third controls the overlap — how many of those active features are shared between them. The figure title updates live with the current Jaccard similarity and Jaccard distance.

The feature grid shows all 20 positions coloured by membership. Purple cells are features present in both vectors — these are the intersection $|A \cap B|$ and form the numerator. Blue cells are present in A but not B; red cells are present in B but not A. Together, purple, blue, and red make up the union $|A \cup B|$ — the denominator. Light grey cells are absent from both vectors. This last category is deliberately muted: Jaccard ignores it entirely, and the annotation panel reports the count of shared absences separately so you can see exactly what is being excluded from the calculation.

To build the key intuition, try this: set both vectors to the same number of features with no overlap, then gradually increase the overlap while keeping everything else fixed. Watch the Jaccard similarity rise from 0 toward 1 purely as a function of shared presence. Then try a different experiment — increase the total number of features in both vectors without changing the overlap. Because the denominator grows while the numerator stays fixed, the similarity falls. This is the behaviour that makes Jaccard appropriate for sparse data: it is not enough to share a few features if each vector contains many others that the other does not.

In [ ]:
# Listing 8.
%matplotlib widget
from visualisations.Figure_7 import show
show()

---

### 5.6 Cosine Similarity

Where Jaccard works with binary presence or absence, **cosine similarity** is designed for frequency or continuous vectors — situations where the raw count or magnitude of each feature matters, not just whether it is non-zero.

To build the right intuition, start with a concrete example. Suppose we have two documents and a vocabulary of just three words: "data", "model", and "learn". We can represent each document as a vector of word counts — one number per word, recording how many times that word appears:

| | "data" | "model" | "learn" |
|---|---|---|---|
| Document A (short summary) | 1 | 2 | 1 |
| Document B (long article) | 3 | 6 | 3 |

Document A is a short summary; document B is a long article on the same topic. Their word counts are different in size, but their *proportions* are identical — document B uses every word exactly three times as often as document A. Intuitively these documents are about the same thing, and a good similarity measure should reflect that.

Geometrically, we can draw each document as an **arrow** (a vector) in a space where each axis represents one word. Document A is the arrow $(1, 2, 1)$ and document B is the arrow $(3, 6, 3)$. Document B's arrow is longer — it points further from the origin because it contains more words overall — but crucially, both arrows point in *exactly the same direction*. If you stood at the origin and looked along document A's arrow, you would be looking directly along document B's arrow too.

Figure 8 below brings this to life. The left panel shows the two document vectors as arrows in word-count space, where the horizontal axis represents the count of the word "data" and the vertical axis represents the count of the word "model". The third word, "learn", contributes to the dot product and norm calculations shown in the annotation panel on the right, but cannot be displayed in a 2D plot — its effect is fully captured in the numbers.

Start with the default settings: Document A = (1, 2, 1) and Document B = (3, 6, 3). Both arrows point in exactly the same direction — B is simply three times longer than A because it is a longer document, but every word appears in the same proportion. The angle between them is $0°$, the cosine similarity is $1.0$, and the cosine distance is $0.0$. The annotation panel on the right shows every step: the dot product, the two norms, their product, and the final division that produces the similarity score.

Now try changing the sliders to break that proportionality. Set Document B to (6, 2, 3) — the word "data" now dominates B much more than it does A. Watch the B arrow swing toward the horizontal axis while A stays fixed, the angle between them opens up, and the cosine similarity drops below 1. The annotation panel updates at each step so you can see exactly which part of the calculation changed.

A few experiments worth trying. First, set both documents to identical counts — the angle collapses to $0°$ and similarity reaches $1.0$. Second, set one document to emphasise early words and the other to emphasise later ones, for example A = (8, 1, 1) and B = (1, 1, 8) — the arrows point in very different directions and similarity falls toward $0$. Third, try scaling one document up uniformly, for example doubling every count in B — the arrow gets longer but does not rotate, and the cosine similarity does not change at all. This last experiment is the clearest demonstration of magnitude-invariance: the metric genuinely does not care how long the document is, only which direction its word proportions point.

In [ ]:
# Listing 9.
%matplotlib widget
from visualisations.Figure_8 import show
show()

Having seen the geometric picture in Figure 8, we can now formalise what cosine similarity is actually computing. The angle between the two arrows is the key quantity — but to calculate it we need two building blocks.

The **dot product** of two vectors $\mathbf{a}$ and $\mathbf{b}$ is computed by multiplying their corresponding elements and summing the results:

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i b_i$$

For our two documents: $\mathbf{a} \cdot \mathbf{b} = (1 \times 3) + (2 \times 6) + (1 \times 3) = 3 + 12 + 3 = 18$. The dot product is large when both vectors have large values in the same positions — exactly what we want to reward. You can verify this in the annotation panel: the first step of the calculation always shows this sum being built term by term.

The **norm** (magnitude) of a vector $\|\mathbf{a}\|$ is its length — the straight-line distance from the origin to the tip of the arrow, computed via the same Pythagorean idea we used for Euclidean distance in Section 5.1:

$$\|\mathbf{a}\| = \sqrt{\sum_{i=1}^{n} a_i^2}$$

For document A: $\|\mathbf{a}\| = \sqrt{1^2 + 2^2 + 1^2} = \sqrt{6} \approx 2.45$. For document B: $\|\mathbf{b}\| = \sqrt{3^2 + 6^2 + 3^2} = \sqrt{54} \approx 7.35$. Document B's arrow is indeed three times longer — but dividing the dot product by both norms cancels that length difference out, leaving only the directional agreement:

$$\cos(\theta) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \, \|\mathbf{b}\|} = \frac{18}{2.45 \times 7.35} \approx \frac{18}{18} = 1.0$$

Exactly 1.0 — the result you saw in Figure 8 when both arrows pointed in the same direction. Two arrows pointing in the same direction have an angle of $0°$ between them, giving $\cos(0°) = 1$. Two arrows pointing in perpendicular directions give $\cos(90°) = 0$. Two arrows pointing in opposite directions give $\cos(180°) = -1$. These three cases map directly onto what you see in the plot when you move the sliders.

This magnitude-invariance is also the metric's main limitation. Two vectors pointing in the same direction always receive a similarity of $1.0$, regardless of how different their magnitudes are. For word-count vectors this is usually the right behaviour — the total length of a document is an accident of how much the author wrote, not a signal about its content. For other data types, such as sensor readings where the scale of a measurement is itself meaningful, ignoring magnitude would be a mistake.

> ⚠️ **Warning:** Cosine similarity is a *similarity* measure, not a distance. For use in distance-based algorithms, convert it via $d_{\cos} = 1 - \cos(\theta)$. Note that this is only a true metric (satisfying the triangle inequality) when all feature values are non-negative — which holds for word counts but not for all data types.

---

### 5.7 How Metric Choice Affects Nearest Neighbours

The metrics we have covered are not interchangeable. Each one defines its own geometry — its own notion of what "close" means — and that geometry directly determines which points an algorithm considers to be neighbours. Switch the metric and you can switch the answer.

The table below makes this concrete. We take a fixed set of eight points and a single query point at $(3, 3)$, then rank the neighbours under three metrics: Euclidean, Manhattan, and Chebyshev. These three are all members of the Minkowski family from Section 5.3, so the comparison is a direct demonstration of how the $p$ parameter changes the geometry rather than just the numbers.

To understand why the rankings can differ, recall the unit ball shapes from Figure 5. Euclidean distance grows equally in all directions — its unit ball is a circle. Manhattan distance grows faster along the diagonals — its unit ball is a diamond, so a point that is diagonally close can appear further away under Manhattan than under Euclidean. Chebyshev distance is dominated by whichever single axis has the largest difference — its unit ball is a square, so two points that are far apart on one axis but close on another can end up equidistant from the origin even if their Euclidean distances are very different.

The practical consequence is that the choice of metric is a modelling decision, not a technical detail. It should be driven by what distance means in your data. If all features are continuous and on comparable scales, Euclidean is a reasonable default. If features are independent and differences along each axis are meaningful individually, Manhattan may be more appropriate. If the worst-case difference across any single feature is what matters — for example in scheduling or logistics problems where the bottleneck dimension is the binding constraint — Chebyshev is the natural choice.

The cell below computes the distance from the query point to every candidate under all three metrics simultaneously and prints the results as a single table. Read it column by column rather than row by row: within each metric's column, find the nearest marker and note which point it lands on. Then compare across columns — if the marker moves to a different row, the metrics are disagreeing about who is closest. The summary beneath the table names the winner under each metric explicitly and flags any disagreements so nothing is lost in the numbers.

In [ ]:
# Listing 10.
# ── Metric sensitivity: does the choice of metric change who is nearest? ──────
# Eight points are arranged so that the nearest neighbour to the query
# genuinely differs depending on which metric is used. Two candidate points
# are placed deliberately: one is diagonally close (favoured by Euclidean),
# the other is close on both axes individually but not diagonally (favoured
# by Chebyshev). This makes the disagreement between metrics visible and
# meaningful rather than a numerical accident.

# The query point sits at the origin. Two candidates are placed so that:
#   Candidate A (1, 3): small x-difference, larger y-difference
#                       → wins under Chebyshev (max diff = 3)
#                         but loses under Euclidean (sqrt(1+9) = 3.16)
#   Candidate B (2, 2): equal differences on both axes
#                       → wins under Euclidean (sqrt(4+4) = 2.83)
#                         but loses under Chebyshev (max diff = 2)
#   Candidate C (0, 4): only y differs
#                       → wins under Manhattan in some configurations
# Additional points fill the space to make the scatter plot look natural.

query = np.array([0.0, 0.0])

points = np.array([
    [ 1.0,  3.0],   # Candidate A — close on x, further on y
    [ 2.0,  2.0],   # Candidate B — equal spread across both axes
    [ 0.0,  4.0],   # Candidate C — only y differs
    [ 3.0,  1.0],   # Candidate D — mirror of A
    [ 4.0,  4.0],   # filler
    [-2.0,  3.0],   # filler
    [ 3.0, -1.0],   # filler
    [-1.0, -3.0],   # filler
], dtype=float)

# Metric definitions — all Minkowski family members for direct comparability
metrics = {
    'Euclidean': lambda a, b: float(np.sqrt(np.sum((a - b) ** 2))),
    'Manhattan': lambda a, b: float(np.sum(np.abs(a - b))),
    'Chebyshev': lambda a, b: float(np.max(np.abs(a - b))),
}

# ── Compute distances from query to every point under every metric ────────────
print('=' * 62)
print(f'  Query point: ({int(query[0])}, {int(query[1])})')
print('=' * 62)
print()

# Column widths chosen so that values, markers, and headers all align
PT_W   = 14
DIST_W = 12
MARK_W = 14
COL_W  = DIST_W + MARK_W

metric_names = list(metrics.keys())
metric_fns   = list(metrics.values())

# Pre-compute distance matrix: rows = points, cols = metrics
dist_matrix = [
    [fn(query, pt) for fn in metric_fns]
    for pt in points
]

# Nearest distance per metric (for marking the winner)
nearest_per_metric = [
    min(row[j] for row in dist_matrix)
    for j in range(len(metrics))
]

# ── Distance table ────────────────────────────────────────────────────────────
print(f'  {"Point":<{PT_W}}' +
      ''.join(f'{n:<{COL_W}}' for n in metric_names))
print('  ' + '─' * (PT_W + len(metrics) * COL_W))

for pt, dists in zip(points, dist_matrix):
    pt_label = f'({int(pt[0]):>2}, {int(pt[1]):>2})'
    row = f'  {pt_label:<{PT_W}}'
    for j, d in enumerate(dists):
        dist_str = f'{d:.3f}'
        marker   = '◀ nearest' if np.isclose(d, nearest_per_metric[j]) else ''
        row += f'{dist_str:<{DIST_W}}{marker:<{MARK_W}}'
    print(row)

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print('  Nearest neighbour under each metric:')
print('  ' + '─' * 50)

winners = {}
for name, fn in metrics.items():
    ranked = sorted(
        (fn(query, pt), (int(pt[0]), int(pt[1])))
        for pt in points
    )
    d, pt = ranked[0]
    winners[name] = pt
    print(f'  {name:<12}  →  {str(pt):<12}  (d = {d:.3f})')

print()

# Highlight any disagreements between metrics
unique_winners = set(winners.values())
if len(unique_winners) == 1:
    print('  All three metrics agree on the nearest neighbour.')
else:
    print('  ⚠  Metrics disagree on the nearest neighbour:')
    for name, pt in winners.items():
        print(f'     {name} says {pt}')
    print()
    print('  The choice of metric is changing the answer,')
    print('  not just the distance value attached to it.')
print()
print('=' * 62)

---

The output above is worth reading carefully. The query point is at the origin $(0, 0)$. Eight candidate points surround it, and for each one we compute the distance under all three metrics simultaneously.

Look first at point $(2, 2)$. It sits exactly two units away on both axes — a perfectly diagonal step from the origin. Under Euclidean distance its straight-line length is $\sqrt{2^2 + 2^2} = 2.828$, which is the shortest straight-line distance of any point in the set, so Euclidean correctly identifies it as the nearest neighbour. Under Chebyshev distance, which only cares about the largest single-axis difference, it scores $\max(2, 2) = 2.0$ — again the smallest in the set, so Chebyshev also picks it.

Now look at the Manhattan column. Point $(2, 2)$ scores $|2| + |2| = 4.0$ — but so do five other points: $(1, 3)$, $(0, 4)$, $(3, 1)$, $(3, -1)$, and $(-1, -3)$. All of them lie on the same Manhattan unit ball — they are all exactly four axis-aligned steps from the origin. When multiple points tie for nearest, the code reports the first one found, which happens to be $(-1, -3)$. The key point is not which specific point wins, but that Manhattan cannot distinguish between any of these five candidates — a cluster that Euclidean and Chebyshev rank very differently from each other.

This happens because of the geometry we saw in Figure 5. The Manhattan unit ball is a diamond: it expands faster along the diagonal directions than along the axes. A point like $(2, 2)$ that sits on the diagonal is *relatively further away* under Manhattan than it appears under Euclidean, because reaching it diagonally costs you steps on two axes simultaneously. Points like $(0, 4)$ or $(3, 1)$ that are aligned more closely with a single axis are pulled in to the same distance.

The summary at the bottom makes the practical consequence explicit: Euclidean and Chebyshev agree that $(2, 2)$ is the nearest neighbour, while Manhattan nominates a different point entirely. If this were a k-NN classifier and the points carried class labels, switching from Euclidean to Manhattan could change the predicted class for the query — not because the data changed, but because the metric changed what "nearest" means.

**Metric quick-reference:**

| Metric | Data type | Sensitive to scale? | Handles correlation? | Notes |
|--------|-----------|--------------------|--------------------|-------|
| Euclidean | Continuous | Yes | No | Default choice; assumes spherical clusters |
| Manhattan | Continuous | Yes | No | Robust to outliers; grid-like paths |
| Minkowski | Continuous | Yes | No | Generalises both; $p$ controls shape |
| Mahalanobis | Continuous | No (scale-invariant) | Yes | Needs enough data to estimate $\mathbf{S}$ |
| Jaccard | Binary / sets | N/A | No | Ignores shared zeros; ideal for text/genomics |
| Cosine | Frequency / sparse | No (magnitude-invariant) | No | Direction only; ideal for NLP |


---

## 6. k-Nearest Neighbours (k-NN)

🔙 [Back to Table of Contents](#table-of-contents)

k-NN is the simplest application of the smoothness assumption: to classify a new point, look at its $k$ nearest training neighbours and take a majority vote. No training phase is required — the entire dataset is the model. This makes k-NN a **lazy learner**: all computation is deferred to prediction time.

The algorithm has three steps:

1. Compute the distance from the query point to every training point.
2. Sort by distance and select the $k$ nearest neighbours.
3. Predict the majority class label among those $k$ neighbours.

The distance metric used in step 1 is not fixed — it is a modelling choice, and everything we covered in Section 5 applies directly here. Euclidean distance is the default in most implementations, including scikit-learn's `KNeighborsClassifier`, and works well when features are continuous, on comparable scales, and the straight-line distance between points is meaningful. If your features are binary, Jaccard distance may be more appropriate. If features have very different variances or are correlated, Mahalanobis distance corrects for that structure. The examples that follow use Euclidean distance throughout, but the `metric` parameter in scikit-learn accepts any of the metrics from Section 5, and changing it can meaningfully change the decision boundary.

**Effect of $k$:** a small value of $k$ produces a very flexible decision boundary that fits closely to the local structure of the training data, at the risk of overfitting to noise. A large value of $k$ averages over more neighbours and produces a smoother boundary, at the risk of underfitting by washing out genuine local structure. This is the bias-variance trade-off in its most direct form — a concept we will return to in detail in Notebook 10.

**Computational cost:** each prediction requires computing distances to all $n$ training points, giving $O(n \cdot d)$ cost per query where $d$ is the number of features. This is manageable for small datasets but becomes expensive at scale — a significant practical limitation that tree-based index structures (such as KD-trees and ball trees) partially address by avoiding exhaustive search.

k-NN inherits all the vulnerabilities of distance metrics. If features are not scaled, high-range features will dominate the distance calculation and the algorithm will effectively ignore the others. Always standardise your features before applying k-NN — the same advice we gave for every distance-based method in Section 5.


### 6.1 From Scratch — Pure Python

Before reaching for scikit-learn, it is worth implementing k-NN by hand. The algorithm is short enough that a from-scratch version fits in a single cell, and reading it makes clear that there is genuinely no training phase — the function simply loops over the training data at prediction time, computes distances, sorts, and votes. Once you have seen that, the scikit-learn version holds no surprises.

The implementation below has two functions. `euclidean_distance` computes the straight-line distance between two feature vectors — the same formula from Section 5.1, now used as the engine of a classifier. `knn_predict` wraps the three algorithmic steps: compute all distances, take the $k$ smallest, return the majority label. Every line maps directly onto the algorithm description above, so you can read the code and the algorithm in parallel.

The dataset is a small two-class problem in 2D, with class 0 clustering in the lower-left and class 1 in the upper-right. The query point is placed deliberately in the ambiguous region between them, so the prediction is genuinely sensitive to the choice of $k$ — a different $k$ consults a different neighbourhood and can reach a different verdict.

The printed output shows the predicted class and the identity of each selected neighbour for $k = 1, 3, 5$. Figure 9 then plots the same information visually, with dashed connector lines from the query to each selected neighbour and a ring around each one so the neighbourhood is unambiguous.

In [ ]:
# Listing 11.
# ── k-NN from scratch ─────────────────────────────────────────────────────────
# Implemented using only NumPy so every algorithmic step is visible.
# scikit-learn's KNeighborsClassifier does the same thing under the hood —
# this version makes the three-step process explicit.

def euclidean_distance(a, b):
    """Straight-line distance between two 1-D feature vectors."""
    return float(np.sqrt(np.sum((a - b) ** 2)))


def knn_predict(X_train, y_train, x_query, k=3):
    """
    Classify x_query by majority vote among its k nearest neighbours.

    Parameters
    ----------
    X_train  : array (n, d)  — training feature matrix
    y_train  : array (n,)    — training class labels
    x_query  : array (d,)    — the point to classify
    k        : int           — number of neighbours to consider

    Returns
    -------
    Predicted class label (the majority vote among the k nearest neighbours).
    """
    # Step 1: compute the Euclidean distance from x_query to every
    # training point and store each distance alongside its label.
    distances = [
        (euclidean_distance(x_query, x_tr), int(y_tr))
        for x_tr, y_tr in zip(X_train, y_train)
    ]

    # Step 2: sort ascending by distance so the nearest neighbours
    # are at the front of the list, then take the first k entries.
    distances.sort(key=lambda tup: tup[0])
    k_nearest = distances[:k]
    k_nearest_labels = [label for _, label in k_nearest]

    # Step 3: majority vote — Counter.most_common(1) returns a
    # [(label, count)] list; index [0][0] extracts just the label.
    return int(Counter(k_nearest_labels).most_common(1)[0][0])


def _pt(arr):
    """
    Convert a NumPy array to a plain-int tuple for clean printing.
    Prevents np.int64 / np.float64 type annotations appearing in output,
    which are implementation details irrelevant to the learner.
    """
    return tuple(int(v) for v in arr)


# ── Dataset ───────────────────────────────────────────────────────────────────
# Two-class dataset in 2D. Class 0 (blue) clusters in the lower-left;
# class 1 (red) clusters in the upper-right. The query point sits in the
# middle, so the prediction is sensitive to the choice of k.
X_knn = np.array(
    [[1, 2], [2, 3], [3, 1], [6, 5], [7, 7],
     [8, 6], [3, 4], [5, 5], [1, 3], [7, 5]],
    dtype=float,
)
y_knn = np.array([0, 0, 0, 1, 1, 1, 0, 1, 0, 1])

x_new = np.array([4.0, 4.0])   # query point — deliberately placed in the
                                # ambiguous region between the two classes

# ── Run predictions for k = 1, 3, 5 and store results for the plot ───────────
k_values = [1, 3, 5]
results  = {}   # k → {'pred': int, 'neighbours': [(dist, idx), ...]}

for k in k_values:
    pred = knn_predict(X_knn, y_knn, x_new, k=k)

    # Store the ranked neighbours so the visualisation can draw the
    # connector lines without recomputing distances.
    ranked = sorted(
        (euclidean_distance(x_new, x), i)
        for i, x in enumerate(X_knn)
    )
    results[k] = {
        'pred':       pred,
        'neighbours': ranked[:k],   # (distance, training index) for each neighbour
    }

# ── Print summary ─────────────────────────────────────────────────────────────
print(f'Query point: {_pt(x_new)}')
print()
print(f'  {"k":<6}  {"Predicted class":<18}  Nearest neighbours')
print('  ' + '─' * 70)

for k in k_values:
    r      = results[k]
    pred   = r['pred']
    neighs = ', '.join(
        f'{_pt(X_knn[idx])} (d={d:.2f})'
        for d, idx in r['neighbours']
    )
    print(f'  k={k:<4}  class {pred:<16}  {neighs}')

---

Figure 9 shows the three panels side by side. The gold star is the query point at $(4, 4)$. Blue points are class 0; red points are class 1. The dashed lines connect the query to whichever training points fall within its neighbourhood for that value of $k$, and each selected neighbour is highlighted with a ring.

With $k = 1$ the algorithm commits entirely to the single closest point, which happens to be class 1 — a confident prediction based on very little evidence. With $k = 3$ the three nearest neighbours are consulted and the majority vote determines the class. With $k = 5$ the neighbourhood widens further, pulling in points from both classes across a larger region of the feature space, and the majority may shift again. The distance table printed below the figure gives the exact distance to each selected neighbour so you can see precisely which points are being included as $k$ grows and by how much the margins change.

Notice that increasing $k$ does not always change the prediction — but when it does, it is because the boundary of the neighbourhood has expanded just far enough to include points of the opposing class that tip the vote. This sensitivity to the exact value of $k$ is most acute in ambiguous regions like this one, where the two classes genuinely intermingle. In regions where one class dominates clearly, the prediction is stable across a wide range of $k$ values.

In [ ]:
# Listing 12.
%matplotlib widget
from visualisations.Figure_9 import show
show(X_knn, y_knn, x_new, k_values, results)

---

### 6.2 k-NN with scikit-learn

The from-scratch implementation in Section 6.1 made every step visible, but it performs an exhaustive distance computation for every prediction — fine for a ten-point toy dataset, not for anything larger. Scikit-learn's `KNeighborsClassifier` implements the same algorithm but adds efficient indexing structures (kd-trees and ball-trees) that avoid computing distances to every training point, as well as built-in support for all the distance metrics from Section 5.

---

#### Creating and training the model

```python
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

# 1. Split the data into training and test sets
#    test_size=0.2 reserves 20% of the data for evaluation
#    random_state fixes the random seed for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Create the model
#    k=5 means the five nearest neighbours vote on the prediction
#    metric='euclidean' uses straight-line distance — this is the
#    parameter you would change to use Manhattan, Mahalanobis, etc.
model = KNeighborsClassifier(n_neighbors=5, metric='euclidean')

# 3. Fit the model — for k-NN this simply stores the training data.
#    There is no parameter estimation here; all computation is deferred
#    to prediction time. This is what "lazy learner" means in practice.
model.fit(X_train, y_train)
```

Unlike the models from Notebooks 5 and 6, `fit()` does not optimise anything. It stores the training data and selects the index structure (kd-tree or ball-tree) that will accelerate neighbour search at prediction time. The real work happens when you call `predict()`.

---

#### Key parameters

| Parameter | Default | What it controls |
|---|---|---|
| `n_neighbors` | `5` | How many neighbours vote — the $k$ in k-NN |
| `metric` | `'minkowski'` | Distance metric used to find neighbours |
| `p` | `2` | Minkowski order: `p=1` gives Manhattan, `p=2` gives Euclidean |
| `weights` | `'uniform'` | `'uniform'` gives each neighbour an equal vote; `'distance'` weights votes by inverse distance so closer neighbours count more |
| `algorithm` | `'auto'` | Index structure for neighbour search: `'kd_tree'`, `'ball_tree'`, or `'brute'` (exhaustive) |
| `n_jobs` | `1` | Number of parallel jobs for neighbour search — set to `-1` to use all available CPU cores |

The `metric` parameter accepts any string recognised by scikit-learn's distance library, including `'euclidean'`, `'manhattan'`, `'chebyshev'`, `'mahalanobis'`, `'cosine'`, and `'jaccard'` — all of which we covered in Section 5. Passing `metric='minkowski'` with `p=2` is exactly equivalent to `metric='euclidean'`; the default settings therefore reproduce the Euclidean k-NN we implemented from scratch.

---

#### Making predictions

```python
# Predict the class label for each test example
#   Returns the class with the most votes among the k nearest neighbours
y_pred = model.predict(X_test)

# Predict the posterior probability of each class for each test example
#   Returns a matrix of shape (n_samples, n_classes)
#   Each row gives the fraction of the k neighbours belonging to each class
y_proba = model.predict_proba(X_test)
```

`predict_proba()` for k-NN returns vote fractions rather than true probabilities — a row of `[0.2, 0.8]` means 80% of the $k$ neighbours belonged to class 1. These are not calibrated probabilities in the statistical sense, but they are useful for ranking predictions by confidence.

---

#### Inspecting the neighbours directly

```python
# Return the distances and indices of the k nearest neighbours
# for each point in X_test — useful for debugging or visualisation
distances, indices = model.kneighbors(X_test)
```

`kneighbors()` exposes the raw neighbour search result. Each row of `distances` gives the distances to the $k$ nearest training points for one test example; each row of `indices` gives their positions in `X_train`. This is how Figure 9 was produced — you can use these arrays to draw the connector lines yourself.

---

#### Evaluating the model

```python
from sklearn.metrics import accuracy_score, classification_report

# Overall fraction of test examples classified correctly
accuracy = accuracy_score(y_test, y_pred)

# Per-class breakdown of precision, recall, and F1
#   (these metrics are covered in depth in Notebooks 10 and 11)
print(classification_report(y_test, y_pred))
```

---

> ⚠️ **Warning:** `KNeighborsClassifier` does not scale its features automatically. If your features have very different ranges, the distance calculation will be dominated by whichever feature has the largest values, exactly as discussed in Section 5.1. Always apply `StandardScaler` or `MinMaxScaler` before fitting a k-NN model. The from-scratch implementation in Section 6.1 has the same vulnerability — it is a property of the algorithm, not the library.

The cell below applies `KNeighborsClassifier` to a larger, three-class synthetic dataset and evaluates performance across a range of $k$ values, showing how accuracy on the test set changes as the neighbourhood size grows.

In [ ]:
# Listing 13.
# k-NN classification using scikit-learn on a three-class synthetic dataset.
# make_blobs generates isotropic Gaussian clusters — cluster_std=1.2 gives
# moderate overlap between classes so the choice of k has a visible effect
# on the decision boundary.
X_sk, y_sk = make_blobs(n_samples=400, centers=3, cluster_std=1.2, random_state=0)

# Reserve 25% of the data for testing — same split used throughout this notebook
X_tr, X_te, y_tr, y_te = train_test_split(X_sk, y_sk, test_size=0.25, random_state=0)

# ── Accuracy vs k ─────────────────────────────────────────────────────────────
# Train a separate k-NN model for each value of k from 1 to 20.
# Recording both train and test accuracy reveals the bias-variance trade-off:
# small k overfits (high train, lower test); large k underfits (both drop).
# metric='euclidean' is made explicit here even though it is the default,
# to connect directly to Section 5.1 and make clear this is a choice.
k_range = range(1, 21)
train_accs, test_accs = [], []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn.fit(X_tr, y_tr)
    train_accs.append(accuracy_score(y_tr, knn.predict(X_tr)))
    test_accs.append(accuracy_score(y_te, knn.predict(X_te)))

# Store the best k for use in the decision boundary plot and the markdown output
best_k        = list(k_range)[np.argmax(test_accs)]
best_test_acc = max(test_accs)
print(f'Best k on test set: k={best_k}  (test accuracy = {best_test_acc:.1%})')

# ── Decision boundary for the best k ─────────────────────────────────────────
# Fit a final model using the best k found above
knn_best = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean')
knn_best.fit(X_tr, y_tr)

# Build a fine grid covering the full feature space. np.meshgrid creates two
# 2D arrays of x and y coordinates; we predict the class at every grid point
# to colour the decision regions.
h   = 0.05   # grid step size — smaller = finer boundary but slower to compute
xlo, xhi = X_sk[:, 0].min() - 1, X_sk[:, 0].max() + 1
ylo, yhi = X_sk[:, 1].min() - 1, X_sk[:, 1].max() + 1
xx, yy   = np.meshgrid(np.arange(xlo, xhi, h), np.arange(ylo, yhi, h))

# np.c_ joins the flattened x and y grids column-wise into a (n_grid_points, 2)
# array that predict() can consume. reshape() maps the flat predictions back
# onto the 2D grid shape so contourf() can draw the filled regions.
Z = knn_best.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

# Left panel: accuracy vs k
# The gap between the train and test curves is the generalisation gap —
# wide at k=1 (overfitting), narrowing as k grows and the boundary smooths.
ax = axes[0]
ax.plot(k_range, train_accs, 'o-', color='steelblue', lw=2, ms=6, label='Train')
ax.plot(k_range, test_accs,  's-', color='tomato',    lw=2, ms=6, label='Test')
ax.axvline(best_k, color='#888', lw=1.2, ls='--',
           label=f'Best k = {best_k}')
ax.set_xlabel('k (number of neighbours)')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs k\nSmall k = flexible boundary;  large k = smooth boundary')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right panel: decision regions for the best k.
# contourf fills each region of the grid with the colour of the predicted class;
# scatter overlays the actual data points so we can see where misclassifications
# occur (points whose colour does not match the background region behind them).
ax = axes[1]
ax.contourf(xx, yy, Z, alpha=0.18, cmap='tab10')

palette_sk = ['steelblue', 'tomato', 'seagreen']
for cls, col in enumerate(palette_sk):
    mask = y_sk == cls
    ax.scatter(
        X_sk[mask, 0], X_sk[mask, 1],
        color=col, s=30, alpha=0.7, edgecolors='k', lw=0.2,
    )

ax.set_title(
    f'Decision regions  (k = {best_k},  metric = Euclidean)\n'
    f'Test accuracy = {best_test_acc:.1%}'
)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')

plt.suptitle(
    'Figure 10: k-NN with scikit-learn — accuracy curve and decision regions',
    fontsize=12, y=0.98,
)
plt.tight_layout()
plt.show()

**When to use k-NN:** small to medium datasets; non-linear decision boundaries; multi-class problems; when interpretability matters (you can show which neighbours drove the prediction).

**When to avoid k-NN:** large datasets ($O(n \cdot d)$ prediction cost); high-dimensional data (the curse of dimensionality makes all distances converge); noisy or irrelevant features (they corrupt the distance).


---

## 7. K-Means Clustering

🔙 [Back to Table of Contents](#table-of-contents)

K-Means is the most widely used clustering algorithm and a natural starting point for unsupervised learning. Given a dataset and a target number of clusters $k$, it partitions every observation into exactly one cluster by repeatedly refining a set of $k$ reference points called **centroids** — one per cluster — until the assignment stabilises.

The objective K-Means minimises is the **Within-Cluster Sum of Squares (WCSS)**, also called **inertia**. For each cluster, we measure how far each of its members lies from the cluster centroid, square those distances, and sum them up. A lower WCSS means tighter, more compact clusters:

$$\text{WCSS} = \sum_{j=1}^{k} \sum_{\mathbf{x}_i \in C_j} \|\mathbf{x}_i - \boldsymbol{\mu}_j\|^2$$

where $C_j$ is the set of points assigned to cluster $j$ and $\boldsymbol{\mu}_j$ is their mean — the centroid of that cluster. Minimising WCSS is equivalent to finding the partition of the data that makes each cluster as internally compact as possible.

The algorithm that achieves this is known as **Lloyd's algorithm**, and it works by alternating between two steps until nothing changes:

1. **Initialise** — place $k$ centroids in the feature space. The simplest approach chooses $k$ data points at random; a smarter initialisation strategy called **k-means++** spreads the initial centroids out deliberately to reduce the chance of a poor solution, and is the default in scikit-learn.
2. **Assign** — assign every data point to its nearest centroid, measured by Euclidean distance. This defines $k$ clusters.
3. **Update** — recompute each centroid as the mean of all the points currently assigned to it. The centroid moves to the centre of its cluster.
4. **Repeat** — return to step 2 with the updated centroids. Points may switch clusters as the centroids move.

Each iteration of steps 2 and 3 is guaranteed to reduce WCSS or leave it unchanged — the algorithm always converges. However, convergence is to a **local minimum**, not necessarily the global one. The final partition depends on where the centroids started, and a bad initialisation can produce a solution that is far from optimal.

This is why scikit-learn's `KMeans` runs the full algorithm `n_init` times with different random initialisations and returns whichever run achieved the lowest WCSS. The default is `n_init=10`, meaning ten independent runs. Increasing it reduces the risk of a poor solution at the cost of computation time.

> ⚠️ **Warning:** K-Means uses Euclidean distance to assign points to centroids, which means it implicitly assumes that clusters are roughly spherical and of similar size. It will struggle with clusters that are elongated, irregularly shaped, or very different in density. DBSCAN and Gaussian Mixture Models, covered later in this notebook, are better suited to those cases.


Before we run K-Means on a real dataset, it is worth building a precise intuition for what WCSS is actually measuring. The formula sums squared distances from every point to its centroid — but what does that mean geometrically, and why is the mean the position that minimises it?

Figure 11 below lets you discover this for yourself. A single cluster of 80 points is shown, and a red centroid star is draggable anywhere on the plot. The thin red lines — called **spokes** — connect every data point to the centroid, and their squared lengths are what WCSS is summing. The annotation panel on the right shows the current WCSS, the minimum achievable WCSS, and how much excess you are carrying above that minimum.

Start by dragging the centroid as far from the data as possible and watch the spokes stretch and WCSS climb. Then drag it back toward the dense centre of the cloud and watch WCSS fall. The green cross marks the true mean of the data — the position that provably minimises WCSS. Try to get the centroid as close to it as you can, and notice that the closer you get, the harder it becomes to reduce WCSS further: the surface is flat near the minimum, which is exactly the convergence behaviour that makes Lloyd's algorithm stable.

This single-cluster picture is the building block of the full K-Means algorithm. When K-Means runs on $k$ clusters simultaneously, it is doing exactly this minimisation for every centroid at once — each centroid trying to sit at the mean of its assigned points, while the assignments themselves change as the centroids move.

In [ ]:
# Listing 14.
%matplotlib widget
from visualisations.Figure_11 import show
show()

### 7.1 From Scratch — Pure Python

As with k-NN in Section 6.1, we implement K-Means by hand before making our lives much easier with scikit-learn. Reading the code alongside the four-step algorithm description (see again below) makes it clear that there is nothing special going on: assigning each point to its nearest centroid is just measuring distances and picking the smallest one; updating a centroid is just computing a mean; convergence is just checking whether anything moved. The entire algorithm fits in a single function.

The implementation introduces one additional thing beyond the bare algorithm: a `history` list that records the centroid positions after every iteration. This lets us trace exactly how the centroids travel through feature space on their way to convergence — a useful diagnostic that scikit-learn does not expose by default.

The four steps of Lloyd's algorithm map directly onto the code:

**Step 1 — Initialise:** `rng_km.choice(len(X), k, replace=False)` picks $k$ data points at random to serve as the starting centroids — one per cluster. Using `replace=False` ensures the same point cannot be chosen twice. These initial positions are a guess; the algorithm will move the centroids from here toward better positions.

**Step 2 — Assign:** the inner loop measures the straight-line distance from each data point to every centroid and records which centroid was closest. After this loop, every point has been allocated to exactly one cluster — whichever centroid it was nearest to.

**Step 3 — Update:** for each cluster, we compute the average position of all the points currently belonging to it and move the centroid there. This is the key insight of K-Means: the position that minimises WCSS for a fixed set of assigned points is always their mean — exactly what Figure 11 showed when you dragged the centroid by hand. The guard `if np.any(labels == j)` handles a rare edge case: if a centroid ended up with no points assigned to it at all, we leave it where it is rather than crashing by taking the mean of an empty set.

**Step 4 — Repeat:** we check whether any centroid moved by more than a tiny threshold ($10^{-6}$) during the last update. If none did, the clusters have stabilised and we stop. Otherwise we go back to step 2 with the new centroid positions — some points may now find themselves closer to a different centroid and switch clusters.

After fitting, the function returns the final cluster labels, centroid positions, WCSS, and this history. The WCSS figure printed at the end is the same quantity from Figure 11 — summed squared distances from every point to its assigned centroid — now evaluated across all $k$ clusters simultaneously rather than just one.


In [ ]:
# Listing 15.
# Rob's K-Means clustering implemented from scratch using only NumPy.
# scikit-learn's KMeans does the same thing with more optimisation —
# this version keeps every step visible.

def kmeans_fit(X, k, n_iters=100, seed=0):
    """
    Fit K-Means to data X using Lloyd's algorithm.

    Parameters
    ----------
    X       : array (n, d)  — the data matrix; n points in d dimensions
    k       : int           — number of clusters to find
    n_iters : int           — maximum number of assign/update cycles before
                              stopping, even if convergence has not been reached
    seed    : int           — random seed; change this to get a different
                              starting configuration and see how sensitive the
                              result is to initialisation

    Returns
    -------
    labels    : array (n,)       — cluster index (0 to k-1) for each data point
    centroids : array (k, d)     — final centroid positions
    wcss      : float            — Within-Cluster Sum of Squares at convergence
    history   : list of arrays   — centroid positions after each iteration,
                                   used by the visualisation to animate movement
    """
    rng_km = np.random.default_rng(seed)

    # ── Step 1: Initialise ────────────────────────────────────────────────────
    # Choose k data points at random as the starting centroids.
    # replace=False ensures no point is chosen twice.
    # astype(float) makes a clean copy so the original data is never modified
    # when we update centroid positions later.
    idx       = rng_km.choice(len(X), k, replace=False)
    centroids = X[idx].copy().astype(float)

    # labels[i] will hold the cluster index (0 to k-1) for each point X[i].
    # Initialised to zero; overwritten on the first assign step.
    labels  = np.zeros(len(X), dtype=int)

    # history keeps a snapshot of centroid positions after each full iteration
    # so the visualisation can replay how they moved toward convergence.
    history = [centroids.copy()]

    for iteration in range(n_iters):

        # ── Step 2: Assign ────────────────────────────────────────────────────
        # For each data point, measure its straight-line distance to every
        # centroid and assign it to the nearest one.
        # This is the most computationally expensive part — O(n * k * d) —
        # which is why scikit-learn uses kd-trees to speed it up.
        for i, x in enumerate(X):
            dists    = [np.sqrt(np.sum((x - c) ** 2)) for c in centroids]
            labels[i] = int(np.argmin(dists))   # index of the nearest centroid

        # ── Step 3: Update ────────────────────────────────────────────────────
        # Move each centroid to the average position of all the points
        # currently assigned to it. As Figure 11 showed, the mean is the
        # position that minimises WCSS for a fixed set of assigned points —
        # so this step always reduces or maintains the overall WCSS.
        #
        # The empty-cluster guard (the else branch) handles a rare but real
        # edge case: if random initialisation placed two centroids so close
        # together that one ends up with zero points after the first assign
        # step, we leave it in place rather than crashing by taking the
        # mean of an empty set.
        new_centroids = np.array([
            X[labels == j].mean(axis=0) if np.any(labels == j) else centroids[j]
            for j in range(k)
        ])

        # ── Step 4: Check for convergence ─────────────────────────────────────
        # If no centroid moved by more than 1e-6 in any direction, the
        # clusters have stabilised and further iterations will not change
        # anything — stop early.
        if np.allclose(centroids, new_centroids, atol=1e-6):
            print(f'  Converged after {iteration + 1} iteration(s)')
            break

        centroids = new_centroids
        history.append(centroids.copy())   # record positions for visualisation

    # ── WCSS ──────────────────────────────────────────────────────────────────
    # Sum of squared distances from every point to its assigned centroid —
    # the same quantity dragged to its minimum in Figure 11, now computed
    # across all k clusters simultaneously.
    wcss = float(sum(
        np.sum((X[labels == j] - centroids[j]) ** 2)
        for j in range(k)
        if np.any(labels == j)
    ))

    return labels, centroids, wcss, history

With our K-Means implementation ready, we can run it on a synthetic dataset and watch the centroids move. We generate the data using `make_blobs` from scikit-learn, which is the standard tool for creating labelled clustering test data. It works by placing a set of Gaussian distributions — one per cluster — at specified locations in feature space and sampling points from each one. The key parameters are:

| Parameter | What it controls |
|---|---|
| `n_samples` | Total number of data points to generate |
| `centers` | Number of cluster centres, or an array of explicit centre coordinates |
| `cluster_std` | Standard deviation of each cluster — smaller values give tighter, more separated clusters; larger values give more overlap |
| `random_state` | Random seed — fix this to get the same dataset every time |

For example, `make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)` generates 200 points drawn from three Gaussian clusters with moderate tightness. `make_blobs` also returns the true cluster labels in `y_true_km` — which we do not pass to `kmeans_fit`, since K-Means is unsupervised, but which we can use afterward to check whether the algorithm recovered the true groupings.

The cell below fits K-Means with $k = 3$ on a 200-point synthetic dataset and prints the number of iterations taken to converge alongside the final WCSS. Figure 12 shows the result: each point coloured by its final cluster assignment, with the converged centroid positions marked as gold stars.

A few things to look for in the plot. The gold stars should sit at the visual centre of mass of their respective clusters — if you covered the colour coding and tried to guess where the centroids should be, you would place them exactly where K-Means has. Points near the boundaries between clusters are the hardest cases: they are genuinely close to two centroids and a slightly different initialisation could assign them to the other cluster without meaningfully changing the WCSS.

Also notice that the cluster colours — blue, red, green — may not correspond to any natural ordering in the data. K-Means assigns cluster indices (0, 1, 2) based purely on which centroid each point ended up nearest to, with no reference to any external labelling. This **label permutation** is normal: cluster 0 in one run might be cluster 2 in another run with a different random seed, even if the underlying partition is identical. When comparing K-Means runs or evaluating against true labels, always match clusters by their content rather than their index.

In [ ]:
# Listing 16.
# ── Generate a synthetic three-cluster dataset ────────────────────────────────
# make_blobs places three Gaussian clusters in 2D feature space.
# cluster_std=0.8 gives moderate separation — enough overlap that the
# boundary regions are interesting, but clear enough that K-Means reliably
# recovers the true groupings.
# random_state=1 fixes the layout so results are reproducible.
X_km, y_true_km = make_blobs(
    n_samples=200, centers=3, cluster_std=0.8, random_state=1
)

# ── Fit K-Means ───────────────────────────────────────────────────────────────
# k=3 because we generated three clusters above. In a real problem we would
# not know the true k — the elbow method in Section 7.2 addresses this.
labels_km, centroids_km, wcss_km, history_km = kmeans_fit(X_km, k=3)

print(f'Converged in {len(history_km)} iteration(s)')
print(f'Final WCSS: {wcss_km:.3f}')

# ── Plot the final clustering ─────────────────────────────────────────────────
# We show only the final result rather than intermediate iterations — the
# centroid movement is covered by Figure 11. Here we want to see the clean
# partition the algorithm settled on.
palette_km = ['steelblue', 'tomato', 'seagreen']

fig, ax = plt.subplots(figsize=(7, 6))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False
# Data points coloured by their final cluster assignment
for j, col in enumerate(palette_km):
    mask = labels_km == j
    ax.scatter(
        X_km[mask, 0], X_km[mask, 1],
        color=col, s=35, alpha=0.7,
        edgecolors='k', lw=0.2,
        label=f'Cluster {j}',
        zorder=2,
    )

# Centroid positions as gold stars — one per cluster
# These are the final positions after convergence, not the starting points.
ax.scatter(
    centroids_km[:, 0], centroids_km[:, 1],
    marker='*', s=320,
    color='gold', edgecolors='black', lw=1.0,
    zorder=4, label='Centroids',
)

ax.set_title(
    f'K-Means final clustering  (k = 3)\n'
    f'WCSS = {wcss_km:.3f}   |   {len(history_km)} iteration(s) to converge',
    fontsize=10,
)
ax.set_xlabel('Feature 1', fontsize=10)
ax.set_ylabel('Feature 2', fontsize=10)
ax.legend(fontsize=9, loc='upper left',
          framealpha=1.0, edgecolor='#cccccc')
ax.grid(True, alpha=0.2)

plt.suptitle(
    'Figure 12: K-Means from scratch — final cluster assignments\n'
    'Gold stars = converged centroid positions',
    fontsize=11, y=0.98,
)
plt.tight_layout()
plt.show()

### 7.2 Choosing k: The Elbow Method

K-Means requires $k$ to be specified before the algorithm runs — but in real unsupervised problems, the whole point is that we do not know the true structure of the data in advance. If we knew there were three clusters, we would not need to cluster at all. So how do we choose $k$?

The simplest answer — try every $k$ value and pick the one with the lowest WCSS — does not work. WCSS always falls as $k$ increases, without exception. This is because adding one more cluster always gives the algorithm an extra centroid to place somewhere, and that extra centroid will always find at least a few points to pull closer to itself, reducing the total squared distance. The more clusters you add, the lower WCSS gets, no matter what the data looks like. Taken to its logical conclusion, setting $k = n$ gives every single point its own private cluster, and WCSS drops to exactly zero because every point is now distance zero from its own centroid. But a model with as many clusters as data points has learned absolutely nothing useful — it has simply given every observation its own label.

What we are looking for is not the lowest WCSS, but the point at which adding more clusters stops buying us meaningful improvement. Up to the true number of clusters, each new centroid captures a genuinely distinct group in the data and WCSS drops sharply. Beyond that point, new centroids are splitting groups that did not need splitting — for example, dividing one natural cluster into two halves — and the gains become small. Plotting WCSS against $k$ produces a curve that bends at this transition, like an elbow in an arm. The value of $k$ at that bend is a reasonable estimate of the true number of clusters.

The elbow is a heuristic, not a guarantee. On real data the curve is often smooth rather than sharply kinked, and reasonable people looking at the same plot may identify different elbows. More rigorous cluster-quality metrics — the silhouette score, the Davies-Bouldin Index, and the Dunn Index — complement and sometimes replace this visual heuristic, and are covered in later notebooks.

---

The cell below applies the elbow method to the same dataset we clustered in Section 7.1. We fit K-Means nine times — once for each value of $k$ from 1 to 9 — and record the WCSS each time. The printed output shows the WCSS at each step so you can see the raw numbers before looking at the curve. Notice how the drops between consecutive values of $k$ get smaller as $k$ increases: the first few clusters capture large, genuine groups and buy large reductions; later clusters are splitting groups that were already reasonably compact, and the gains shrink accordingly.

Figure 13 plots these values as a curve. The drop-off annotations between each pair of consecutive $k$ values make the diminishing returns visible numerically as well as visually. The dashed red line marks $k = 3$ — the elbow — and the shaded region to its right shows where additional clusters are no longer earning their place. Because we generated this data with `make_blobs` using three centres, we know the true answer is $k = 3$, and the elbow correctly identifies it. In a real problem we would not have this ground truth, and the elbow would be our primary guide.

In [ ]:
# Listing 17.
# ── Elbow method: fit K-Means for k = 1 to 9 and record WCSS ─────────────────
# We reuse kmeans_fit from Section 7.1 on the same dataset (X_km) so the
# elbow plot is directly comparable to the clustering shown in Figure 12.
# seed=42 is fixed across all runs so differences in WCSS across k values
# reflect genuine structural differences, not lucky or unlucky initialisations.
k_range_e = range(1, 10)
wcss_vals  = []

for k in k_range_e:
    _, _, wcss_k, _ = kmeans_fit(X_km, k=k, seed=42)
    wcss_vals.append(wcss_k)
    print(f'  k={k}   WCSS = {wcss_k:.3f}')

# ── Identify the elbow ────────────────────────────────────────────────────────
# We know the true k is 3 because we generated the data with make_blobs.
# In a real problem this would be unknown — we would identify the elbow
# visually from the plot below.
elbow_k = 3

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

ax.plot(
    list(k_range_e), wcss_vals,
    'o-', color='steelblue', lw=2, ms=8,
    label='WCSS',
)

# Annotate the drop between each consecutive k so the learner can see
# exactly where the gains become small
for i in range(1, len(wcss_vals)):
    drop = wcss_vals[i - 1] - wcss_vals[i]
    ax.annotate(
        f'−{drop:.0f}',
        xy=(i + 1, wcss_vals[i]),
        xytext=(i + 1 + 0.15, wcss_vals[i] + (wcss_vals[0] * 0.03)),
        fontsize=7.5, color='#555555',
    )

# Vertical dashed line marking the elbow — the point where the drops
# become small and adding more clusters yields diminishing returns
ax.axvline(
    elbow_k, color='tomato', lw=1.8,
    linestyle='--', label=f'Elbow at k = {elbow_k}',
)

# Shade the region to the right of the elbow to show where gains are small
ax.axvspan(
    elbow_k, max(k_range_e),
    alpha=0.06, color='tomato',
    label='Diminishing returns',
)

ax.set_xlabel('Number of clusters k', fontsize=10)
ax.set_ylabel('WCSS (Within-Cluster Sum of Squares)', fontsize=10)
ax.set_title(
    'Elbow method — WCSS falls as k increases\n'
    'The sharpest bend occurs at k = 3, matching the true number of clusters',
    fontsize=10,
)
ax.set_xticks(list(k_range_e))
ax.legend(fontsize=9, framealpha=1.0, edgecolor='#cccccc')
ax.grid(True, alpha=0.25)

plt.suptitle(
    'Figure 13: Elbow method — choosing k by looking for diminishing returns in WCSS',
    fontsize=11, y=1.02,
)
plt.tight_layout()
plt.show()

### 7.3 K-Means with scikit-learn

The from-scratch implementation in Section 7.1 made every step of Lloyd's algorithm visible, but it has two practical limitations: it uses simple random initialisation, which is sensitive to bad starting positions, and it computes distances in a plain Python loop, which is slow on large datasets. Scikit-learn's `KMeans` addresses both these issues — it uses **k-means++ initialisation** by default, which spreads the initial centroids deliberately across the data to reduce the chance of a poor solution, and runs the full algorithm `n_init` times, keeping whichever run achieved the lowest WCSS.

---

#### Creating and fitting the model

Here's how we can fit a k-means model in scikit-learn.

```python
from sklearn.cluster import KMeans

# Create the model.
# n_clusters is the only required parameter — the k in K-Means.
# init='k-means++' spreads the initial centroids across the data rather
# than choosing them entirely at random, reducing sensitivity to bad starts.
# n_init=10 runs the full algorithm 10 times with different initialisations
# and keeps the result with the lowest WCSS — the default and a safe choice
# for most datasets.
# random_state fixes the random seed so results are reproducible.
model = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)

# Fit the model — this runs the full assign/update cycle n_init times
# and stores the best result. Unlike k-NN, real computation happens here.
model.fit(X_train)
```

Note that `fit()` takes only `X` — there are no labels `y`. This is unsupervised learning: the algorithm has no target to optimise toward, only the geometry of the data.

---

#### Key parameters

| Parameter | Default | What it controls |
|---|---|---|
| `n_clusters` | `8` | The number of clusters $k$ — the only parameter you must always set explicitly |
| `init` | `'k-means++'` | Initialisation strategy: `'k-means++'` spreads centroids out deliberately; `'random'` picks data points at random (same as our from-scratch version) |
| `n_init` | `10` | Number of independent runs with different initialisations — the best result by WCSS is kept |
| `max_iter` | `300` | Maximum assign/update cycles per run before stopping, even if convergence has not been reached |
| `tol` | `1e-4` | Convergence threshold — the run stops when centroids move less than this between iterations |
| `random_state` | `None` | Random seed — fix this for reproducibility |

The most important parameter by far is `n_clusters`. Everything else has sensible defaults for most problems. If you find K-Means producing poor results, increasing `n_init` is the first thing to try — more restarts reduce the chance of a bad local minimum.

---

#### Accessing the fitted model

After calling `fit()`, the trained model exposes its results as attributes you can inspect directly:

```python
model.labels_          # cluster index (0 to k-1) for each training point
model.cluster_centers_ # array of shape (k, d) — the final centroid positions
model.inertia_         # final WCSS (the best across all n_init runs)
model.n_iter_          # number of iterations the best run took to converge
```

`inertia_` is scikit-learn's name for WCSS — the same quantity we minimised in Figure 11 and tracked in the elbow plot in Figure 13.

---

#### Predicting cluster membership for new points

```python
# Assign new points to their nearest centroid
# Returns the cluster index for each point in X_new
cluster_ids = model.predict(X_new)

# fit_predict combines fit and predict in one call — useful when you
# want the labels for the training data itself
labels = model.fit_predict(X)
```

Unlike supervised models, there is no notion of accuracy here — the predicted cluster index for a new point is simply the index of the nearest centroid, measured by Euclidean distance.

---

#### Evaluating cluster quality

```python
from sklearn.metrics import silhouette_score

# Silhouette score: ranges from -1 (poor) to +1 (perfect separation)
# Higher is better — a score above 0.5 generally indicates well-defined clusters
score = silhouette_score(X, model.labels_)
print(f'Silhouette score: {score:.3f}')
```

Silhouette score and other cluster quality metrics are covered in depth in Notebook 12. For now, treat it as a complement to the elbow plot: a high silhouette score at your chosen $k$ is a good sign that the clusters are genuinely compact and well-separated.

---

> ⚠️ **Warning:** Like the from-scratch version, scikit-learn's `KMeans` uses Euclidean distance and assumes clusters are roughly spherical and of similar size. It will reliably find compact, blob-shaped clusters but will struggle with elongated shapes, rings, or clusters of very different densities. If your problem requires a different distance measure — for example, Manhattan distance for high-dimensional sparse data, or a custom domain-specific metric — consider `KMedoids` from `scikit-learn-extra`, or DBSCAN, both of which accept arbitrary distance metrics.

We have built and tested our own K-Means implementation, so now let us see how it compares to the version built into scikit-learn. Both receive identical data that we generated earlier, and the same $k = 3$, so any difference in WCSS comes purely from the initialisation strategy and the number of restarts.

The from-scratch version uses simple random initialisation and runs once. It will find a reasonable solution most of the time on clean synthetic data, but a single unlucky starting configuration can leave it in a local minimum that is noticeably worse than the best available partition. Scikit-learn's version uses k-means++ initialisation — which places starting centroids as far apart from each other as possible rather than choosing them at random, reducing the chance of two centroids starting in the same region of the data. It then runs the full algorithm ten times from different starting positions and keeps whichever run achieved the lowest WCSS.

On clean, well-separated synthetic data like `X_km` the two versions will often find the same solution, because the clusters are distinct enough that almost any starting configuration leads to the right partition. The difference becomes more apparent on messier real-world data where clusters overlap, vary in density, or are not spherical — precisely the conditions where a bad initialisation is more likely to trap the algorithm in a poor local minimum.

Run the code below to view the comparison. The printed output below the plot reports both WCSS values and the percentage improvement scikit-learn achieves over the from-scratch version, so you can see directly whether the more sophisticated initialisation made a difference on this particular dataset.

In [ ]:
# Listing 18.
# ── Compare from-scratch K-Means with scikit-learn's implementation ───────────
# Both are fitted on the same dataset (X_km) with k=3.
# The comparison shows the practical benefit of k-means++ initialisation
# and multiple restarts: scikit-learn's version will match or beat the
# from-scratch WCSS because it runs 10 independent attempts and keeps
# the best, while our version runs once from a single random starting point.

km_sk = KMeans(
    n_clusters=3,
    init='k-means++',   # smarter initialisation than pure random — spreads
                        # starting centroids across the data deliberately
    n_init=10,          # run the full algorithm 10 times; keep the best result
    random_state=0,     # fixed seed for reproducibility
)
labels_sk = km_sk.fit_predict(X_km)

# ── Print comparison ──────────────────────────────────────────────────────────
print('K-Means comparison on the same dataset:')
print()
print(f'  From-scratch (random init, 1 run):')
print(f'    WCSS              = {wcss_km:.3f}')
print(f'    Iterations        = {len(history_km)}')
print()
print(f'  scikit-learn (k-means++ init, 10 runs):')
print(f'    WCSS (inertia)    = {km_sk.inertia_:.3f}')
print(f'    Iterations (best) = {km_sk.n_iter_}')
print()

# A lower WCSS in the sklearn version means it found a tighter partition.
# The difference may be small on clean synthetic data — on messier real
# data it can be substantial.
if km_sk.inertia_ < wcss_km:
    improvement = 100 * (wcss_km - km_sk.inertia_) / wcss_km
    print(f'  scikit-learn achieved {improvement:.1f}% lower WCSS than the from-scratch version.')
else:
    print(f'  Both versions found the same solution on this dataset.')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

configs = [
    (axes[0], labels_km,  centroids_km,            wcss_km,
     'From-scratch K-Means\n(random init, 1 run)'),
    (axes[1], labels_sk,  km_sk.cluster_centers_,  km_sk.inertia_,
     f'scikit-learn KMeans\n(k-means++ init, {km_sk.n_init} runs)'),
]

for ax, labels, centroids, wcss, title in configs:

    # Data points coloured by cluster assignment
    for j, col in enumerate(palette_km):
        mask = labels == j
        ax.scatter(
            X_km[mask, 0], X_km[mask, 1],
            color=col, s=35, alpha=0.7,
            edgecolors='k', lw=0.2,
            label=f'Cluster {j}',
            zorder=2,
        )

    # Centroid stars — gold so they stand out against all cluster colours
    ax.scatter(
        centroids[:, 0], centroids[:, 1],
        marker='*', s=300,
        color='gold', edgecolors='black', lw=1.0,
        zorder=4, label='Centroids',
    )

    ax.set_title(f'{title}\nWCSS = {wcss:.3f}', fontsize=10)
    ax.set_xlabel('Feature 1', fontsize=10)
    ax.set_ylabel('Feature 2', fontsize=10)
    ax.legend(fontsize=8, loc='upper left',
              framealpha=1.0, edgecolor='#cccccc')
    ax.grid(True, alpha=0.2)

plt.suptitle(
    'Figure 14: From-scratch vs scikit-learn K-Means on the same dataset\n'
    'k-means++ initialisation and multiple restarts reduce the risk of a poor solution',
    fontsize=11, y=0.98,
)
plt.tight_layout()
plt.show()

---

The previous comparison used clean, balanced, well-separated clusters where almost any starting configuration leads to the right answer. To see where random initialisation actually fails, we need a harder dataset — one where the odds of drawing a bad starting configuration are high enough that a single random run is likely to get it wrong.

The dataset below has three clusters but they are not balanced: one contains 200 points and two contain only 30 each. The large cluster dominates the feature space, so a random draw of three starting centroids is more likely to place two of them inside the large cluster than to spread them evenly. When that happens, the algorithm splits the large cluster in two and merges the two small clusters into one — a partition that looks locally stable but is far from the true grouping. Scikit-learn's k-means++ initialisation avoids this by actively pushing each new starting centroid away from the ones already chosen, making it far less likely that two centroids begin in the same region.

In [ ]:
# Listing 19.
# ── A harder dataset designed to expose the weakness of random initialisation ─
# The custom version uses a single random start, which means a bad draw can
# place two starting centroids inside the same natural cluster and leave the
# third cluster unrepresented at the start. This is most likely to happen
# when clusters are close together and of unequal density.
#
# We create three clusters with:
#   - very different sizes (n_samples per cluster)
#   - tighter spread than X_km (cluster_std=0.6) so they are compact but close
#   - centres chosen to sit near each other, increasing the chance that
#     random initialisation places two starting points in the large cluster
#
# We also fix a seed for kmeans_fit that is known to produce a bad start
# on this particular dataset, so the difference is reliably visible.

X_hard, _ = make_blobs(
    n_samples=[200, 30, 30],   # one large cluster, two small ones —
                                # random init will often place two centroids
                                # in the large cluster and miss a small one
    centers=[[-2, 0], [2, 1], [2, -1]],   # small clusters are close together,
                                            # making them easy to confuse
    cluster_std=0.6,
    random_state=7,
)

# ── Fit both versions ─────────────────────────────────────────────────────────
# seed=3 is chosen because random initialisation on this dataset with this
# seed places two starting centroids inside the large cluster — a reliably
# bad start that scikit-learn's k-means++ avoids.
labels_hard_scratch, centroids_hard_scratch, wcss_hard_scratch, _ = kmeans_fit(
    X_hard, k=3, seed=3
)

km_hard = KMeans(
    n_clusters=3,
    init='k-means++',
    n_init=10,
    random_state=0,
)
labels_hard_sk = km_hard.fit_predict(X_hard)

# ── Print comparison ──────────────────────────────────────────────────────────
print('K-Means comparison on the harder dataset:')
print()
print(f'  From-scratch (random init, seed=3, 1 run):')
print(f'    WCSS = {wcss_hard_scratch:.3f}')
print()
print(f'  scikit-learn (k-means++ init, 10 runs):')
print(f'    WCSS = {km_hard.inertia_:.3f}')
print()

if km_hard.inertia_ < wcss_hard_scratch:
    improvement = 100 * (wcss_hard_scratch - km_hard.inertia_) / wcss_hard_scratch
    print(f'  scikit-learn achieved {improvement:.1f}% lower WCSS.')
    print(f'  The from-scratch version was trapped in a local minimum.')
else:
    print(f'  Both versions found the same solution on this dataset.')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.canvas.header_visible  = False
fig.canvas.toolbar_visible = False

configs = [
    (axes[0], labels_hard_scratch, centroids_hard_scratch, wcss_hard_scratch,
     'From-scratch K-Means\n(random init, seed=3, 1 run)'),
    (axes[1], labels_hard_sk,      km_hard.cluster_centers_, km_hard.inertia_,
     f'scikit-learn KMeans\n(k-means++ init, {km_hard.n_init} runs)'),
]

for ax, labels, centroids, wcss, title in configs:

    for j, col in enumerate(palette_km):
        mask = labels == j
        ax.scatter(
            X_hard[mask, 0], X_hard[mask, 1],
            color=col, s=35, alpha=0.7,
            edgecolors='k', lw=0.2,
            label=f'Cluster {j}',
            zorder=2,
        )

    ax.scatter(
        centroids[:, 0], centroids[:, 1],
        marker='*', s=300,
        color='gold', edgecolors='black', lw=1.0,
        zorder=4, label='Centroids',
    )

    ax.set_title(f'{title}\nWCSS = {wcss:.3f}', fontsize=10)
    ax.set_xlabel('Feature 1', fontsize=10)
    ax.set_ylabel('Feature 2', fontsize=10)
    ax.legend(fontsize=8, loc='lower left',
              framealpha=1.0, edgecolor='#cccccc')
    ax.grid(True, alpha=0.2)

plt.suptitle(
    'Figure 15: Random initialisation fails on an unbalanced dataset\n'
    'The from-scratch version places two centroids in the large cluster and misses a small one',
    fontsize=11, y=0.98,
)
plt.tight_layout()
plt.show()

**Strengths of K-Means:** simple and fast; scales to large datasets; interpretable centroids; partitions space cleanly.

**Weaknesses of K-Means:** $k$ must be specified in advance; sensitive to random initialisation (mitigated by `n_init`); assumes spherical, roughly equal-sized clusters; cannot find non-convex cluster shapes.

## 8. Hierarchical Clustering

🔙 [Back to Table of Contents](#table-of-contents)

K-Means forces you to decide how many clusters you want before the algorithm runs. The elbow method helps, but it still requires fitting the model repeatedly and making a judgement call about where the curve bends. For a newcomer to a dataset, this feels backwards: you are being asked to answer the very question you are trying to investigate before you have looked at the data properly.

Hierarchical clustering sidesteps this entirely. Instead of committing to a number of clusters upfront, it runs once and produces a complete picture of how the data connects at every level of granularity simultaneously. You can then inspect that picture and decide where to cut — how many clusters to extract — after you have seen the structure of the data rather than before. Think of it like zooming in and out on a map: at a high zoom level you see a few large regions; zoom in further and each region splits into smaller sub-regions. Hierarchical clustering gives you the whole map, and you choose the zoom level that makes sense for your problem.

The most common variant is **agglomerative** clustering, which builds this picture from the bottom up. Every point starts as its own cluster. The algorithm then repeatedly finds the two clusters that are closest together and merges them into one, recording the distance at which the merge happened. This continues until all points have been absorbed into a single cluster. The record of all these merges — who merged with whom, and at what distance — is displayed as a **dendrogram**: a tree diagram where the height of each branching point on the vertical axis shows how far apart two clusters were when they joined.

The dendrogram is the key output. A tall vertical line in the dendrogram means the algorithm had to reach a long way to find the next merge — the clusters at that level are genuinely far apart and well-separated. A short vertical line means the two clusters that merged were already very similar. When you see a large gap between two successive merge heights, that gap is the natural place to cut: it is the data telling you that below this point the clusters are tight and coherent, and above it you are forcing things together that do not naturally belong. Drawing a horizontal line across the dendrogram at that gap, and counting how many vertical branches it crosses, gives you both the number of clusters and a sense of how confident you should be in that number.

The algorithm itself is straightforward:

1. Start with $n$ clusters, one per data point.
2. Compute the distance between every pair of clusters.
3. Merge the two closest clusters into one.
4. Recompute distances and repeat from step 3.
5. Stop when all points belong to a single cluster.

The critical design choice is the **linkage criterion** — the rule used to define the distance between two clusters, given that a cluster contains multiple points rather than just one. Different linkage criteria produce very different cluster shapes:

| Linkage | Distance between clusters | Tends to produce | Watch out for |
|---|---|---|---|
| Single | Smallest distance between any pair of points across the two clusters | Long, chain-like clusters that follow the shape of the data | Very sensitive to outliers — one stray point can chain two clusters together |
| Complete | Largest distance between any pair of points across the two clusters | Compact, roughly equal-sized clusters | Can split large natural clusters to enforce compactness |
| Average | Mean distance across all pairs of points between the two clusters | A balance between single and complete | Less extreme than either, but less interpretable |
| Ward | Merges the pair of clusters that produces the smallest increase in total WCSS | Compact, spherical clusters similar to K-Means | Sensitive to outliers; assumes roughly equal cluster sizes |

Ward linkage is generally the best default choice when clusters are expected to be roughly spherical and balanced — it produces results similar to K-Means but without requiring $k$ in advance. Single linkage is the right choice when you expect clusters to follow irregular, elongated shapes. Complete linkage is a reasonable middle ground when you want compact clusters but do not want to assume spherical geometry.

### 8.1 Linkage Criteria

It is worth building a precise understanding of what each of these criteria is actually measuring. The four linkage methods all answer the same question — how far apart are these two clusters? — but they answer it in fundamentally different ways, and that difference determines the shape of the clusters they produce.

Figure 15a below makes this concrete with the simplest possible setup: two small clusters, **A (blue)** and **B (red)**, each containing four draggable points. Every point in both clusters can be moved freely, and as you drag, four distance measurements update live:

The **green solid line** shows the **Single linkage** distance — the length of the shortest possible line between any point in A and any point in B. Single linkage only cares about the closest pair across the gap. It is very sensitive to outliers: drag one point from cluster A toward cluster B and watch the single linkage distance collapse, even if the rest of both clusters remain far apart. This is why single linkage tends to produce long chain-like clusters — one stray point can bridge two groups that are otherwise well-separated.

The **orange dashed line** shows the **Complete linkage** distance — the length of the longest line between any point in A and any point in B. Complete linkage only cares about the most extreme pair. It is resistant to the outlier effect but can be overly conservative: drag one point from A far away from B and the complete linkage distance grows even if the bulk of the two clusters are actually quite close. This is why complete linkage tends to produce compact, roughly equal-sized clusters — it refuses to merge groups until even their most distant members are reasonably close.

The **dotted black line** shows the **Ward linkage** distance — drawn between the two cluster centroids (marked with + symbols). Ward does not directly measure the distance between individual points at all. Instead it measures the increase in total WCSS that would result from merging the two clusters: how much tighter would the combined cluster's members need to sit around a single centroid compared to sitting around their own separate centroids? Drag the two clusters close together and the Ward distance falls sharply; spread them apart and it rises. Ward linkage is the most sensitive to overall cluster compactness rather than to any individual pair of points, which is why it tends to produce the most balanced and spherical clusters.

The annotation panel on the right shows the numeric value of all four distances simultaneously so you can read off exactly how much they diverge as you reshape the clusters.

Try these experiments to build the key intuitions. First, drag one point from cluster A to sit very close to cluster B, leaving the rest of A far away — watch Single linkage fall dramatically while Complete and Ward barely change. Second, stretch cluster A into an elongated shape by pulling one point far from the others — watch Complete linkage grow while Single is unaffected. Third, move both clusters as a whole toward each other without changing their internal shape — watch all four distances fall together, but at different rates. The ratio between Single and Complete distances tells you how elongated the gap between the clusters is; the Ward distance tells you how much merging them would cost in terms of cluster compactness.

In [ ]:
# Listing 20.
%matplotlib widget
from visualisations.Figure_15a import show
show()

### 8.2 Dendrogram - then what?
Once you have the dendrogram in front of you, the next step is to choose a cut height — a horizontal line drawn across the tree that determines how many clusters you extract. Every vertical branch that your horizontal line crosses becomes one cluster, containing all the data points below that branch. Move the line upward and branches merge together, giving fewer, larger clusters. Move it downward and branches split apart, giving more, smaller clusters.

In practice you are looking for the largest vertical gap in the dendrogram — the level where the merge distances suddenly jump. If the data contains four genuine clusters, the dendrogram will show three relatively short merges at the bottom (points joining nearby neighbours) and then a noticeably larger jump when the algorithm is forced to bridge the gap between the four natural groups. Draw your horizontal line just below that jump, and you recover four clusters. The size of the gap is effectively a measure of confidence: a large gap means the clusters at that level are clearly separated; a small gap means the boundary between that number of clusters and one fewer is ambiguous.

This is what makes hierarchical clustering genuinely useful for exploratory analysis. Rather than committing to $k = 4$ upfront and hoping the data agrees, you run the algorithm once, look at the dendrogram, and let the structure of the data guide you to a sensible number of clusters. If the dendrogram shows one obvious large gap, the choice of $k$ is clear. If it shows several gaps of similar size, the data may not have a single natural granularity and you have learned something important about its structure — something the elbow method alone would not have told you.

Once you have chosen a cut height, `AgglomerativeClustering` from scikit-learn assigns a cluster label to every data point for that specific $k$, which you can then use for downstream analysis exactly as you would use the labels from K-Means. The difference is that your choice of $k$ was informed by the data rather than guessed in advance.

---

Figure 16 below puts the dendrogram to work interactively. The left panel shows the full Ward-linkage dendrogram for an 80-point dataset; the right panel shows the cluster assignments that result from cutting the tree at the current height. The red dashed line is the cut — click anywhere on the dendrogram panel to move it, or click and drag.

Start by placing the cut line near the top of the dendrogram, where only one or two large merges have been made. The scatter plot will show one or two coarse clusters covering the whole dataset. Now drag the line downward slowly. Each time it crosses a branching point in the dendrogram, one cluster splits into two and the scatter plot updates. Watch for the largest vertical gap in the dendrogram — the level where two branches are furthest apart before they merge. Cutting there recovers the natural structure of the data, and you should see the four underlying clusters emerge cleanly on the right. Continue dragging the line all the way to the bottom and watch the clusters fragment into noise as the algorithm is forced to treat every minor local variation as a meaningful group.

This is the core skill of hierarchical clustering in practice: reading the dendrogram to find the level at which the data separates most naturally, rather than imposing a number of clusters from outside.

In [ ]:
# Listing 21.
%matplotlib widget
from visualisations.Figure_16 import show
show()

### 8.3 Hierarchical Clustering with scikit-learn

Scipy's `linkage` and `dendrogram` functions, used in Figure 16, are the right tools for exploring the full tree and choosing a cut height visually. Once you have decided on a value of $k$, scikit-learn's `AgglomerativeClustering` assigns cluster labels to every data point for that specific cut — and integrates cleanly with the rest of the scikit-learn ecosystem.

---

#### Creating and fitting the model

```python
from sklearn.cluster import AgglomerativeClustering

# Create the model.
# n_clusters is the number of clusters to extract — equivalent to choosing
# a cut height on the dendrogram that produces exactly this many groups.
# linkage='ward' minimises the increase in WCSS at each merge and is the
# best default for compact, roughly spherical clusters.
model = AgglomerativeClustering(n_clusters=4, linkage='ward')

# Fit the model and retrieve cluster labels in one step.
# Like KMeans, fit() takes only X — there are no labels y.
labels = model.fit_predict(X)
```

There is no separate `predict()` call for new points — `AgglomerativeClustering` is a transductive method, meaning it assigns labels only to the data it was fitted on. If you need to classify new points after fitting, use `KMeans` or train a supervised classifier on the cluster labels produced here.

---

#### Key parameters

| Parameter | Default | What it controls |
|---|---|---|
| `n_clusters` | `2` | Number of clusters to extract — equivalent to a horizontal cut across the dendrogram |
| `linkage` | `'ward'` | Linkage criterion: `'ward'`, `'complete'`, `'average'`, or `'single'` — see the table in Section 8 |
| `metric` | `'euclidean'` | Distance metric used to compute pairwise distances between points. Note: `'ward'` linkage only works with Euclidean distance; for other metrics use `'complete'`, `'average'`, or `'single'` |
| `compute_full_tree` | `'auto'` | Whether to build the full tree beyond `n_clusters` — needed if you want to cut at a different level later |
| `connectivity` | `None` | Optional connectivity matrix constraining which points can be merged — useful for spatially structured data such as images |

The most important choice after `n_clusters` is `linkage`. Ward is the right default for most tabular data. If your clusters are elongated or follow irregular shapes, try `'single'` or `'average'`. If you need a metric other than Euclidean, you must use `'complete'`, `'average'`, or `'single'` — Ward linkage is mathematically defined only for Euclidean distance.

---

#### Accessing the fitted model

```python
model.labels_       # cluster index (0 to n_clusters-1) for each data point
model.n_clusters_   # number of clusters found (same as n_clusters parameter)
model.n_leaves_     # number of leaves in the tree (= number of data points)
model.n_connected_components_   # number of connected components in the graph
```

---

#### Using scipy to choose the cut height first

If you want to use the dendrogram to choose $k$ before fitting scikit-learn's model, the typical workflow is:

```python
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import matplotlib.pyplot as plt

# Step 1: build the full linkage matrix with scipy
Z = linkage(X, method='ward')

# Step 2: plot the dendrogram and identify the largest gap visually
fig, ax = plt.subplots(figsize=(10, 5))
dendrogram(Z, ax=ax, no_labels=True)
plt.show()

# Step 3: cut the tree at the chosen height to get labels directly from scipy
#   criterion='distance' cuts at a fixed height; t is the cut height
labels = fcluster(Z, t=10.0, criterion='distance') - 1  # subtract 1 for 0-based labels

# Alternatively, cut to get exactly k clusters
labels = fcluster(Z, t=4, criterion='maxclust') - 1
```

This is exactly the workflow Figure 16 is built on — scipy builds and cuts the tree, scikit-learn assigns the labels for downstream use.

---

#### Evaluating cluster quality

```python
from sklearn.metrics import silhouette_score

score = silhouette_score(X, labels)
print(f'Silhouette score: {score:.3f}')
```

As with K-Means, silhouette score is the most practical quick check on cluster quality. A score above 0.5 generally indicates well-defined, well-separated clusters. Silhouette score and the Davies-Bouldin and Dunn indices are covered in depth in Notebook 12.

> ⚠️ **Warning:** Agglomerative clustering has a computational cost of $O(n^2)$ in memory and $O(n^2 \log n)$ in time — it needs to compute and store the distance between every pair of points. This is fine for datasets up to a few thousand points but becomes impractical for large datasets. If you are working with tens of thousands of points or more, K-Means or DBSCAN are more scalable alternatives.

---

Figure 17 below shows all three linkage methods on the same 80-point dataset simultaneously. The top row shows the full dendrogram for each method; the bottom row shows the cluster assignments that result from cutting each tree at the current cut depth. A single draggable cut line moves proportionally across all three dendrograms at once — drag it on any panel and all six panels update together.

Each dendrogram uses its own y-axis scale so no tree is compressed. The cut line moves as a fraction of each tree's own height, meaning the same relative depth is applied to all three simultaneously. This is important because the three linkage methods produce merge distances on very different scales — Ward's distances reflect changes in WCSS and are not directly comparable to Single's nearest-neighbour distances — so forcing a shared absolute scale would make one or two of the trees unreadable.

The three columns are deliberately aligned so you can compare them side by side. Ward linkage produces a balanced, symmetrical tree with a clear large gap near the top — drag the cut line to just below that gap and the four underlying clusters emerge cleanly with sharp boundaries. Single linkage produces a very different tree: long chains of short merges punctuated by occasional larger ones, which leads to elongated clusters that follow the shape of the data rather than imposing spherical boundaries. Try dragging to the same relative depth on the Single dendrogram and notice how differently the data has been partitioned. Complete linkage sits between the two — more compact than Single but less balanced than Ward, and more sensitive to points at the edges of clusters.

When reading the dendrograms, focus on the vertical gaps between successive merge heights. A large gap means the algorithm had to reach much further to find the next pair of clusters to join — the groups at that level are genuinely well-separated and the cut belongs there. A series of small gaps means many merges are happening at similar distances — there is no strong natural boundary in that region and any choice of $k$ there is somewhat arbitrary. The Ward dendrogram on this dataset shows one obvious large gap; the Single linkage dendrogram shows a much less clean picture, illustrating why linkage choice matters as much as the cut height itself.

In [ ]:
# Listing 22.
%matplotlib widget
from visualisations.Figure_17 import show
show()

**Strengths:** deterministic; the dendrogram gives an interpretable multi-resolution view of cluster structure; flexible distance measures.

**Weaknesses:** $O(n^2)$ to $O(n^3)$ time complexity — impractical for large datasets; merge decisions are irreversible; sensitive to noise and outliers.


---


## 9. DBSCAN

🔙 [Back to Table of Contents](#table-of-contents)

Every clustering algorithm we have covered so far — K-Means and hierarchical clustering — works by measuring distance to a centroid or to other points, and both implicitly assume that clusters are compact and roughly convex in shape. Real data frequently violates this assumption. Consider a dataset of GPS coordinates tracing the paths of pedestrians through a city: the natural clusters are elongated corridors, not spherical blobs. Or consider a dataset where one dense population of points forms a ring around another — K-Means will always split these into wedge-shaped sectors rather than recovering the ring, because no centroid placement can capture a circular structure.

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) takes a completely different approach. Instead of asking "which centroid is this point closest to?", it asks "is this point in a dense region of the data?" A cluster, under DBSCAN's definition, is a contiguous region of high point density. Two points belong to the same cluster if you can walk from one to the other through a chain of densely packed neighbours, regardless of the overall shape that chain traces out. This means DBSCAN can recover clusters of any shape — rings, crescents, spirals, or irregular blobs — as long as the density within the cluster is higher than the density between clusters.

A second major advantage is that DBSCAN does not require $k$ to be specified in advance. The number of clusters emerges naturally from the density structure of the data. Points that sit in sparse regions — too far from any dense group to be part of one — are automatically labelled as **noise** rather than being forced into the nearest cluster.

To understand how DBSCAN works, we need to define three types of point:

* A **core point** is a point that has at least `minPts` other points within a radius of $\varepsilon$ around it. It sits in a genuinely dense region.
* A **border point** sits within $\varepsilon$ of a core point but does not have enough neighbours of its own to qualify as a core point — it is on the edge of a dense region rather than inside it.
* A **noise point** is neither a core point nor within reach of one. It sits in a sparse region that belongs to no cluster, and scikit-learn labels it $-1$.

Figure 17a below gives you a hands-on feel for what $\varepsilon$ and `minPts` actually do on a small, controlled dataset. Adjust the sliders and watch how the three point types — core, border, and noise — shift in response. The dataset contains two tight clusters, one loose cluster, and a handful of isolated outliers, so the effect of tightening or relaxing each parameter is immediately visible. Once the behaviour feels intuitive, Figure 19 will make more sense.

In [ ]:
# Listing 23.
%matplotlib widget
from visualisations.Figure_17a import show
show()

---

The DBSCAN algorithm proceeds as follows:

1. Pick any unvisited point. If it has at least `minPts` neighbours within radius $\varepsilon$, declare it a core point and start a new cluster.
2. Expand the cluster outward by recursively visiting all points within $\varepsilon$ of the current core point. Any of those that are themselves core points expand the cluster further; border points are added to the cluster but do not expand it.
3. Continue until no more density-reachable points remain. The cluster is complete.
4. Move to the next unvisited point and repeat. Any point that is never reached by this process is labelled as noise.

The entire algorithm is controlled by just two parameters: $\varepsilon$, the radius of the neighbourhood around each point, and `minPts`, the minimum number of points that neighbourhood must contain for the point to be considered a core point. Together they define what "dense" means for your dataset. A small $\varepsilon$ or a large `minPts` makes the density threshold strict — fewer points qualify as core points, more points are labelled noise, and only the densest regions form clusters. A large $\varepsilon$ or a small `minPts` relaxes the threshold — more points are pulled into clusters and fewer are left as noise, but at the risk of merging clusters that should remain separate.

> ⚠️ **Warning:** DBSCAN has no notion of a centroid and makes no assumption about cluster shape, but it does assume that all clusters have roughly similar density. If your data contains clusters of very different densities — one very tight group and one loose sprawling group — a single pair of $\varepsilon$ and `minPts` values may correctly identify one cluster while either splitting the other into fragments or merging it with noise. OPTICS, covered in Notebook 9, addresses this limitation by building a density profile across a range of $\varepsilon$ values simultaneously.

---

Figure 18 below makes these three point types visible and interactive. The dataset is two interleaved crescents — a shape that K-Means cannot recover but DBSCAN handles naturally. Two sliders control $\varepsilon$ and `minPts`, and as you adjust them every point is reclassified live:

Filled circles are **core points** — they have at least `minPts` neighbours within radius $\varepsilon$ and sit in genuinely dense regions. Diamond markers are **border points** — they sit within $\varepsilon$ of a core point but do not have enough neighbours of their own to be core. Cross markers are **noise points** — they sit in sparse regions that no core point can reach. When the $\varepsilon$ circles are visible, each circle shows the neighbourhood of one core point: every other point falling inside that circle counts toward its neighbour total.

The annotation panel on the right shows the current counts of core, border, and noise points alongside the number of clusters found, so you can track exactly how the composition changes as you move the sliders.

Start with $\varepsilon = 0.30$ and `minPts = 5` — the default settings. You should see the two crescents recovered as separate clusters with a small number of noise points at the tips where the data is thinnest. Now try the following experiments to build intuition for each parameter.

**Increasing $\varepsilon$:** as the neighbourhood radius grows, more points fall within each core point's circle, so more points qualify as core. At the same time, the circles begin to overlap across the gap between the two crescents, eventually connecting them into a single cluster. Watch for the moment the two clusters merge — this is the point where $\varepsilon$ has grown large enough to bridge the gap between the densest regions of each crescent.

**Decreasing $\varepsilon$:** as the radius shrinks, fewer points fall within each neighbourhood. Points that were core become border points, then noise. The clusters fragment from the outside in — the tips of the crescents, where the data is thinnest, are the first to be reclassified as noise.

**Increasing `minPts`:** raising the density threshold makes the algorithm stricter. More points in the interior of the crescents lose their core status and become border points or noise, and the clusters thin down to only the densest regions.

**Decreasing `minPts`:** lowering the threshold to 2 or 3 makes almost every point a core point and pulls nearly all noise into clusters. The algorithm becomes very permissive — useful when data is genuinely sparse but prone to connecting regions that should be separate.

Use the checkbox to toggle the $\varepsilon$ circles on and off. With many core points the circles overlap and clutter the plot; turning them off gives a cleaner view of the cluster assignments. Turn them back on when you want to understand why a specific point was classified as core or border.


In [ ]:
# Listing 24.
%matplotlib widget
from visualisations.Figure_18 import show
show()

### 9.1 DBSCAN with scikit-learn

DBSCAN is one of the simplest algorithms to apply in scikit-learn — it has only two parameters that matter and requires no decisions about the number of clusters.

---

#### Creating and fitting the model

```python
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

# Standardise features before fitting — DBSCAN uses distance measurements
# directly, so unscaled features will cause high-range variables to dominate
# the neighbourhood calculation exactly as they do in k-NN and K-Means.
X_scaled = StandardScaler().fit_transform(X)

# Create the model.
# eps     : the neighbourhood radius ε — points within this distance of a
#           core point are considered its neighbours
# min_samples : the minimum number of neighbours (minPts) a point must have
#               within eps to qualify as a core point
model = DBSCAN(eps=0.5, min_samples=5)

# Fit the model and retrieve cluster labels in one step.
# Like AgglomerativeClustering, there is no separate predict() call.
labels = model.fit_predict(X_scaled)
```

There is no `predict()` method for new points — DBSCAN assigns labels only to the data it was fitted on. If you need to classify new points after fitting, the standard approach is to train a k-NN classifier on the DBSCAN labels and use that for prediction.

---

#### Key parameters

| Parameter | Default | What it controls |
|---|---|---|
| `eps` | `0.5` | The neighbourhood radius $\varepsilon$ — the maximum distance between two points for one to be considered a neighbour of the other |
| `min_samples` | `5` | The minimum number of points within `eps` of a point for it to be classified as a core point — equivalent to `minPts` |
| `metric` | `'euclidean'` | Distance metric used to measure neighbourhoods — accepts any metric from Section 5, including `'manhattan'`, `'cosine'`, and custom functions |
| `algorithm` | `'auto'` | Index structure for neighbour search: `'ball_tree'`, `'kd_tree'`, or `'brute'` — `'auto'` selects the most appropriate one |
| `n_jobs` | `1` | Number of parallel jobs for the neighbour search — set to `-1` to use all available cores |

The two parameters that matter in practice are `eps` and `min_samples`. Everything else has sensible defaults. Choosing good values for `eps` and `min_samples` is the main challenge with DBSCAN — see the parameter selection guidance below.

---

#### Reading the output

```python
# Cluster labels for each point — -1 means noise
print(model.labels_)

# The indices of all core points in the training data
print(model.core_sample_indices_)

# A copy of the core point feature vectors themselves
print(model.components_)

# Count how many clusters were found (excluding noise)
n_clusters = len(set(labels) - {-1})
n_noise    = (labels == -1).sum()
print(f'Clusters found: {n_clusters}')
print(f'Noise points:   {n_noise}')
```

The label $-1$ is scikit-learn's convention for noise — points that were not reachable from any core point. These are not assigned to any cluster and should be treated separately in downstream analysis. Do not pass them to a silhouette scorer without filtering them out first, as the silhouette score is undefined for noise points.

---

#### Choosing eps and min_samples

Selecting good parameter values is the most practically challenging part of using DBSCAN. Two approaches are commonly used together:

**The k-distance plot** is the most reliable heuristic for choosing `eps`. For each point, compute the distance to its $k$-th nearest neighbour (where $k$ = `min_samples`), sort these distances in ascending order, and plot them. The curve will show a sharp elbow — the point at which distances start rising steeply is a natural choice for `eps`, because it marks the transition from dense to sparse regions.

```python
from sklearn.neighbors import NearestNeighbors

# Fit a nearest-neighbour model to find the distance to the kth neighbour
# for every point. k = min_samples is the standard choice.
k   = 5
nn  = NearestNeighbors(n_neighbors=k).fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)

# The kth neighbour distance for each point is the last column
kth_distances = np.sort(distances[:, -1])

plt.plot(kth_distances)
plt.xlabel('Points sorted by distance')
plt.ylabel(f'Distance to {k}th nearest neighbour')
plt.title('k-distance plot — look for the elbow')
plt.grid(True, alpha=0.3)
plt.show()
```

**`min_samples`** is less sensitive than `eps` and a value of 2×(number of features) is a common starting point for low-dimensional data. For 2D data, `min_samples=4` or `5` is typical. Larger values make the algorithm more robust to noise but require denser clusters to form.

---

#### Evaluating cluster quality

```python
from sklearn.metrics import silhouette_score

# Filter out noise points before scoring — silhouette score is undefined
# for points with no cluster assignment
mask  = labels != -1
score = silhouette_score(X_scaled[mask], labels[mask])
print(f'Silhouette score (excluding noise): {score:.3f}')
```

---

> ⚠️ **Warning:** DBSCAN scales as $O(n^2)$ with a brute-force neighbour search, though the kd-tree and ball-tree algorithms reduce this to $O(n \log n)$ for low-dimensional data. For very large datasets — hundreds of thousands of points in high dimensions — DBSCAN can become slow regardless of the algorithm choice. In those cases, consider HDBSCAN (available in scikit-learn from version 1.3) which uses a more efficient density hierarchy and is generally more robust to parameter choice.

Figure 19 puts DBSCAN and K-Means head to head on two datasets that are specifically designed to break K-Means. The left column shows what K-Means produces; the right column shows what DBSCAN recovers with the same data.

The top row uses two interleaved crescents. K-Means, forced to place two centroids somewhere in the feature space, draws a straight boundary between them — the only kind of boundary it can draw. The result splits each crescent in half rather than separating them from each other. DBSCAN ignores centroids entirely and instead traces the density of the data, following each crescent as a continuous dense region and correctly identifying them as two separate clusters.

The bottom row uses a ring of points surrounding a smaller central cluster. This is a case where K-Means cannot possibly succeed regardless of initialisation: the centroid of the outer ring sits at the centre of the feature space, in exactly the same location as the centroid of the inner cluster. No assignment of two centroids can separate a ring from the point it surrounds. DBSCAN again recovers the true structure cleanly, because the gap between the ring and the inner cluster is a low-density region and DBSCAN treats any such gap as a cluster boundary.

In [ ]:
# Listing 25.
%matplotlib widget
from visualisations.Figure_19 import show
show()

**Strengths of DBSCAN:** discovers clusters of arbitrary shape; automatically identifies noise/outlier points; does not require $k$ to be specified; deterministic.

**Weaknesses of DBSCAN:** sensitive to the choice of $\varepsilon$ and `minPts`; struggles with clusters of widely varying density; distance metrics lose meaning in high-dimensional spaces.



---

## 10. Summary

🔙 [Back to Table of Contents](#Table-of-Contents)


**Clustering algorithms covered in this notebook:**

| Algorithm | Type | Key parameter(s) | Best for | Limitations |
|-----------|------|-----------------|----------|-------------|
| k-NN | Lazy / neighbourhood | $k$ | Simple, interpretable, non-linear boundaries | Slow at prediction time; suffers from the curse of dimensionality |
| K-Means | Centroid-based | $k$ | Fast, scalable, well-separated spherical clusters | $k$ must be pre-specified; assumes convex shapes; local minima |
| Hierarchical | Tree / agglomerative | Linkage criterion, cut height | No $k$ needed; interpretable dendrogram | $O(n^3)$ complexity; not scalable to large datasets |
| DBSCAN | Density-based | $\varepsilon$, `minPts` | Arbitrary cluster shapes; automatic noise detection | Parameter sensitive; struggles with varying cluster density |

**Choosing a distance metric:**

| Situation | Recommended metric |
|-----------|-------------------|
| General continuous data with similar scales | Euclidean |
| Continuous data; outlier-robust or grid-like movement | Manhattan |
| Continuous data with correlations or differing scales | Mahalanobis |
| Binary or set-valued features | Jaccard |
| Text or sparse vectors where magnitude is irrelevant | Cosine |


---

## 11. References

🔙 [Back to Table of Contents](#Table-of-Contents)


Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.

Ester, M., Kriegel, H.-P., Sander, J., & Xu, X. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. In *Proceedings of the 2nd International Conference on Knowledge Discovery and Data Mining (KDD-96)* (pp. 226–231). AAAI Press.

MacQueen, J. (1967). Some methods for classification and analysis of multivariate observations. In *Proceedings of the 5th Berkeley Symposium on Mathematical Statistics and Probability*, Vol. 1 (pp. 281–297). University of California Press.

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.

Virtanen, P., Gommers, R., Oliphant, T. E., et al. (2020). SciPy 1.0: Fundamental algorithms for scientific computing in Python. *Nature Methods*, 17(3), 261–272. https://doi.org/10.1038/s41592-019-0686-2

Ward, J. H. (1963). Hierarchical grouping to optimize an objective function. *Journal of the American Statistical Association*, 58(301), 236–244.